# Repository management

> Inspect and change a Git repository through one locked gateway, with recovery points before risky operations.

`gheasy.repo` contains the local Git operations used by agents and editor interfaces. It moved from `ramabana.git`; that module now re-exports this API so existing Leela imports continue to work.

The design has three rules.

1. Every Git process runs through `GitGateway`. Reads can run together. Writes are exclusive. Network commands use a separate lock and do not delay local status reads.
2. Every guarded mutation records a `Safepoint` first. The safepoint stores `HEAD`, the current branch, and uncommitted work. `undo` restores it.
3. Merge and rebase previews use `git merge-tree`. They create temporary Git objects without changing refs, the index, or the working tree.

Read the notebook from top to bottom. Each implementation cell is followed by executable checks against a temporary repository.

In [ ]:
#| default_exp repo

In [ ]:
#| export
from __future__ import annotations

import difflib, json, re, shlex, shutil, subprocess, threading, time
from contextlib import contextmanager
from dataclasses import asdict, dataclass
from urllib.parse import urlparse

from fastcore.all import L, Path, first, patch, uniqueify

In [ ]:
#| hide
import atexit, os, shutil as _shutil, tempfile
from fastcore.test import test_eq, test_ne, test_fail

_tmp = []
atexit.register(lambda: [_shutil.rmtree(d, ignore_errors=True) for d in _tmp])

def sh(cwd, *args):
    "Run Git directly when preparing a test fixture."
    return subprocess.run(['git', *args], cwd=str(cwd), check=True, text=True,
                          capture_output=True).stdout.strip()

def mkrepo(files=None, name='work'):
    "Create a temporary repository with one commit and a local author identity."
    d = Path(tempfile.mkdtemp())/name
    d.mkdir()
    _tmp.append(str(d.parent))
    sh(d, 'init', '-b', 'main')
    sh(d, 'config', 'user.email', 'tests@example.com')
    sh(d, 'config', 'user.name', 'Repo tests')
    sh(d, 'config', 'commit.gpgsign', 'false')
    for k, v in (files or {'readme.md': 'hello\n'}).items():
        (d/k).parent.mkdir(parents=True, exist_ok=True)
        (d/k).write_text(v)
    sh(d, 'add', '-A')
    sh(d, 'commit', '-m', 'initial')
    return d

def mkbare(name='origin.git'):
    "Create a local bare remote for network-command tests."
    d = Path(tempfile.mkdtemp())/name
    _tmp.append(str(d.parent))
    sh(d.parent, 'init', '--bare', name)
    sh(d, 'symbolic-ref', 'HEAD', 'refs/heads/main')
    return d

def write(repo, path, text):
    (repo/path).parent.mkdir(parents=True, exist_ok=True)
    (repo/path).write_text(text)
    return repo/path

def commit(repo, message, **files):
    for k, v in files.items(): write(repo, k.replace('__', '.'), v)
    sh(repo, 'add', '-A')
    sh(repo, 'commit', '-m', message)
    return sh(repo, 'rev-parse', '--short', 'HEAD')

## Classifying commands

Every command is classified before it runs. `read` commands use a shared lock and pass `--no-optional-locks`. `write` commands use an exclusive lock. `net` commands use a separate lock, which lets local reads continue during a slow fetch.

Unknown command forms are writes. This reduces concurrency but prevents an unrecognised mutation from running beside another operation.

In [ ]:
#| export
class GitError(RuntimeError): pass

#: `commit-tree` writes an object and touches no ref, index or worktree. A loose object is
#: written atomically, so a rehearsal that builds hundreds of them needs no write lock.
READS = frozenset({
    'blame', 'cat-file', 'check-attr', 'check-ignore', 'commit-tree', 'count-objects', 'describe',
    'diff', 'diff-index', 'diff-tree', 'for-each-ref', 'grep', 'log', 'ls-files', 'ls-tree',
    'merge-base', 'merge-tree', 'name-rev', 'reflog', 'rev-list', 'rev-parse', 'shortlog',
    'show', 'show-ref', 'status', 'var', 'verify-commit', 'whatchanged',
})

NET = frozenset({'clone', 'fetch', 'ls-remote', 'push'})

MIXED = {
    'branch': frozenset({'--show-current', '--list', '-l', '--contains', '--no-contains', '--merged',
        '--no-merged', '--points-at', '--format', '-a', '--all', '-r', '--remotes', '-v', '-vv', '--verbose'}),
    'config': frozenset({'--get', '--get-all', '--get-regexp', '--list', '-l'}),
    'notes': frozenset({'list', 'show'}),
    'remote': frozenset({'get-url', 'show', '-v', '--verbose'}),
    'stash': frozenset({'list', 'show'}),
    'submodule': frozenset({'status', 'summary'}),
    'tag': frozenset({'-l', '--list', '--contains', '--no-contains', '--points-at', '--format', '--sort'}),
    'worktree': frozenset({'list'}),
}

BARE_READS = frozenset({'branch', 'remote', 'tag'})

def classify(args):
    "Classify a Git argument list as `read`, `write`, or `net`; unknown forms are writes."
    rest = [str(a) for a in args]
    while rest and rest[0].startswith('-'):
        if rest[0] in ('-c', '-C', '--git-dir', '--work-tree', '--namespace') and len(rest) > 1: rest = rest[2:]
        else: rest = rest[1:]
    if not rest: return 'write'
    sub, rest = rest[0], [a for a in rest[1:] if a]
    if sub in NET: return 'net'
    if sub in READS: return 'read'
    if sub == 'symbolic-ref': return 'write' if len([a for a in rest if not a.startswith('-')]) > 1 else 'read'
    if sub in MIXED:
        if not rest: return 'read' if sub in BARE_READS else 'write'
        return 'read' if rest[0].partition('=')[0] in MIXED[sub] else 'write'
    return 'write'

In [ ]:
test_eq(classify(('status', '--porcelain=v1')), 'read')
test_eq(classify(('add', '--', 'x.py')), 'write')
test_eq(classify(('fetch', 'origin')), 'net')
test_eq(classify(('branch',)), 'read')                  # a bare `branch` lists
test_eq(classify(('branch', '-d', 'x')), 'write')
test_eq(classify(('branch', '--show-current')), 'read')
test_eq(classify(('config', '--get=x')), 'read')        # `--opt=value` is read by its stem
test_eq(classify(('config', 'user.name', 'x')), 'write')
test_eq(classify(('worktree', 'list')), 'read')
test_eq(classify(('worktree', 'add', 'x')), 'write')
test_eq(classify(('symbolic-ref', 'HEAD')), 'read')     # one argument reads, two write
test_eq(classify(('symbolic-ref', 'HEAD', 'refs/heads/x')), 'write')
test_eq(classify(('-C', '/tmp', 'status')), 'read')     # a global option, then the subcommand
test_eq(classify(('--no-pager', 'log')), 'read')
test_eq(classify(()), 'write')                          # unreadable, so pessimistic
test_eq(classify(('sneak',)), 'write')
test_eq(classify(('commit-tree', 'x')), 'read')         # writes an object, touches no ref

## Locking one repository

`RepoLock` is a readers-writer lock. The writing thread can acquire the write lock again and can read while it holds the lock. This lets a mutation inspect its own result without deadlocking.

Network commands use a separate lock because they do not modify the index or working tree.

In [ ]:
#| export
class RepoLock:
    "A per-repository readers-writer lock, reentrant for the thread holding the write."
    def __init__(self):
        self._cv = threading.Condition()
        self._writer = None
        self._depth = 0
        self._readers = 0
        self._net = threading.RLock()
    
    @contextmanager
    def read(self):
        me = threading.get_ident()
        with self._cv:
            while self._writer is not None and self._writer != me: self._cv.wait()
            self._readers += 1
        try: yield
        finally:
            with self._cv:
                self._readers -= 1
                if not self._readers: self._cv.notify_all()
    
    @contextmanager
    def write(self):
        me = threading.get_ident()
        with self._cv:
            if self._writer == me: self._depth += 1
            else:
                while self._writer is not None or self._readers: self._cv.wait()
                self._writer, self._depth = me, 1
        try: yield
        finally:
            with self._cv:
                self._depth -= 1
                if not self._depth:
                    self._writer = None
                    self._cv.notify_all()
    
    @contextmanager
    def net(self):
        with self._net: yield

In [ ]:
lock = RepoLock()
with lock.read():
    with lock.read(): pass                              # readers do not exclude each other
with lock.write():
    with lock.write(): pass                             # the writing thread is reentrant
    with lock.read(): pass                              # ...and may read inside its own write

seen = []
def reader():
    with lock.read(): seen.append('read')
with lock.write():
    t = threading.Thread(target=reader); t.start(); t.join(.2)
    test_eq(seen, [])                                   # a writer excludes a reader in another thread
t.join(2)
test_eq(seen, ['read'])

held = threading.Event()
def netter():
    with lock.net(): held.set(); time.sleep(.05)
t = threading.Thread(target=netter); t.start()
held.wait(2)
with lock.read(): pass                                  # ...but the network lock does not
t.join(2)

## Recovery state

Before a guarded mutation, a `Safepoint` records the current commit, branch, and uncommitted work. Its refs and journal stay inside `.git` and are never pushed.

`SAFEPOINT_REF` and `JOURNAL_NAME` retain the name `leela` for compatibility with existing repositories. Changing these names would make earlier safepoints unreachable.

In [ ]:
#| export
FOREIGN_LOCK = re.compile(r"(?:index\.lock|\.lock'?: File exists|Another git process seems to be running|Unable to create '[^']*\.lock')", re.I)
LOCK_ATTEMPTS, LOCK_BACKOFF,  = 4, 0.15
SAFEPOINT_REF = 'refs/leela/safepoint'
JOURNAL_NAME, JOURNAL_KEEP = 'leela-safepoints.json', 60

@dataclass
class Safepoint:
    "Where a repository was, immediately before one mutation."
    token: str
    op: str
    head: str
    branch: str
    stash: str
    dirty: int
    created_at: int
    def describe(self):
        where = self.branch or (self.head[:9] + ' (detached)' if self.head else 'an unborn branch')
        return f'{self.op} on {where}' + (f', {self.dirty} uncommitted file(s) kept' if self.dirty else '')

In [ ]:
p = Safepoint(token='a1', op='merge', head='abcdef1234', branch='main', stash='', dirty=0, created_at=0)
test_eq(p.describe(), 'merge on main')
test_eq(Safepoint('a1', 'rebase', 'abcdef1234', '', '', 0, 0).describe(), 'rebase on abcdef123 (detached)')
test_eq(Safepoint('a1', 'commit', '', '', '', 0, 0).describe(), 'commit on an unborn branch')
test_eq(Safepoint('a1', 'merge', 'abcdef1234', 'main', 'x', 2, 0).describe(),
        'merge on main, 2 uncommitted file(s) kept')

assert FOREIGN_LOCK.search("fatal: Unable to create '/r/.git/index.lock': File exists")
assert FOREIGN_LOCK.search('Another git process seems to be running in this repository')
assert not FOREIGN_LOCK.search('fatal: not a git repository')

## The Git gateway

One `GitGateway` instance owns the process-wide repository locks. Callers must share this instance for locking to work.

The gateway also caches repository environments and other values that change less often than the working tree. An application can set `env_for` to provide a checkout-specific environment. When `env_for` is `None`, Git inherits the process environment.

In [ ]:
#| export
class GitGateway:
    "Run Git with shared repository locks, cached environments, and recovery journals."
    def __init__(self):
        self._locks = {}
        self._registry = threading.Lock()
        self._env_cache = {}
        self._exe = None
    def _lock(self, root):
        key = str(root)
        with self._registry:
            if key not in self._locks: self._locks[key] = RepoLock()
            return self._locks[key]
    def _git(self):
        if self._exe is None: self._exe = shutil.which('git') or ''
        if not self._exe: raise GitError('git is not installed or not on PATH')
        return self._exe
    env_for = None   #: Optional checkout-specific environment provider
    def env(self, cwd, ttl=30):
        "Return the checkout environment supplied by `env_for`."
        if self.env_for is None: return None
        key = str(cwd)
        hit = self._env_cache.get(key)
        if hit and time.monotonic() - hit[0] < ttl: return hit[1]
        value = self.env_for(cwd)
        self._env_cache[key] = (time.monotonic(), value)
        return value
    def memo(self, cwd, key, make, ttl=60):
        "Cache a repository value for `ttl` seconds."
        cache_key = (str(cwd), key)
        hit = self._env_cache.get(cache_key)
        if hit and time.monotonic() - hit[0] < ttl: return hit[1]
        value = make()
        self._env_cache[cache_key] = (time.monotonic(), value)
        return value
    def resolves(self, cwd, command):
        "Resolve the executable used by a configured filter or hook."
        try: parts = shlex.split(str(command or ''))
        except ValueError: return ''
        if not parts: return ''
        return shutil.which(parts[0], path=(self.env(cwd) or {}).get('PATH')) or ''

In [ ]:
gw = GitGateway()
r = mkrepo()
test_eq(gw._lock(r) is gw._lock(r), True)               # one lock per root, shared across callers
test_ne(gw._lock(r), gw._lock(mkrepo()))
assert gw._git().endswith('git')

calls = []
def make(): calls.append(1); return len(calls)
test_eq(gw.memo(r, 'k', make), 1)
test_eq(gw.memo(r, 'k', make), 1)                       # inside the ttl, the answer is not remade
test_eq(gw.memo(r, 'k', make, ttl=-1), 2)               # ...and outside it, it is
test_eq(gw.memo(r, 'other', make), 3)                   # keyed by (root, key)

test_eq(gw.env(r), None)                                # no `env_for` wired in: inherit the process
gw.env_for = lambda cwd: {'PATH': '/nowhere'}
test_eq(gw.env(r), {'PATH': '/nowhere'})
test_eq(gw.resolves(r, 'jupyter'), '')                  # not on that PATH
test_eq(gw.resolves(r, ''), '')
test_eq(gw.resolves(r, 'a "b'), '')                     # unparseable, not a crash
gw.env_for = None

## Running a command

`GitGateway.run` is the only function in this module that starts Git. It acquires the lock selected by `classify`, then calls `_exec`. `_exec` retries failures caused by another process holding a Git lock.

A failed checked command raises `GitError`. The message includes stderr and stdout because Git can put useful information on either stream.

In [ ]:
#| export
@patch
def run(self:GitGateway, cwd, *args, check=True, input=None, timeout=30, kind=None):
    "Run one Git command under its repository lock."
    kind = kind or classify(args)
    lock = self._lock(cwd)
    with {'read': lock.read, 'write': lock.write, 'net': lock.net}[kind]():
        return self._exec(cwd, args, check=check, input=input, timeout=timeout, kind=kind)

@patch
def _exec(self:GitGateway, cwd, args, check, input, timeout, kind):
    argv = [self._git(), *(['--no-optional-locks'] if kind == 'read' else []), *map(str, args)]
    for attempt in range(LOCK_ATTEMPTS):
        try:
            p = subprocess.run(
                argv, cwd=str(cwd), input=input, text=True,
                encoding='utf-8', errors='replace', stdout=subprocess.PIPE,
                stderr=subprocess.PIPE, timeout=timeout, env=self.env(cwd),
            )
        except subprocess.TimeoutExpired as e:
            raise GitError(f'git {args[0] if args else "command"} timed out after {timeout}s') from e
        if not p.returncode or attempt == LOCK_ATTEMPTS - 1: break
        if not FOREIGN_LOCK.search(p.stderr or ''): break
        time.sleep(LOCK_BACKOFF * 2 ** attempt)
    if check and p.returncode:
        message = '\n'.join(x for x in ((p.stderr or '').strip(), (p.stdout or '').strip())
                            if x) or f'git exited {p.returncode}'
        if FOREIGN_LOCK.search(message):
            message += ('\n\nAnother Git process is using this repository -- a terminal, an '
                'agent, or an editor elsewhere. Nothing was changed.')
        raise GitError(message)
    return p

@patch
def out(self:GitGateway, cwd, *args, **kwargs):
    "Run Git and return stdout."
    return self.run(cwd, *args, **kwargs).stdout

In [ ]:
gw, r = GitGateway(), mkrepo()
test_eq(gw.out(r, 'branch', '--show-current').strip(), 'main')
test_eq(gw.run(r, 'status', '--porcelain=v1').stdout, '')
test_eq(gw.run(r, 'rev-parse', 'nope', check=False).returncode != 0, True)   # check=False reports
test_fail(lambda: gw.run(r, 'rev-parse', 'nope'), contains='nope')           # ...and check=True raises
test_fail(lambda: gw.run(r, 'checkout', 'nope'), contains='did not match')

write(r, 'readme.md', 'changed\n')
test_eq(gw.out(r, 'status', '--porcelain=v1').rstrip(), ' M readme.md')
test_eq(gw.run(r, 'add', '-A', kind='write').returncode, 0)                  # an explicit kind wins
test_eq(gw.out(r, 'status', '--porcelain=v1').rstrip(), 'M  readme.md')

# stdin reaches the command, which is how a patch is applied.
test_eq(gw.run(r, 'hash-object', '-w', '--stdin', input='x\n').stdout.strip()[:2].isalnum(), True)
test_fail(lambda: gw.run(r, 'log', '--format=%H', timeout=0), contains='timed out')

## Recording recovery points

The journal is a JSON file inside `.git`. Entries are newest first and limited to `JOURNAL_KEEP`. A temporary file and atomic rename prevent readers from seeing a partial write.

Journal read and write errors do not block a Git mutation. An unreadable journal is treated as empty.

In [ ]:
#| export
@patch
def _git_dir(self:GitGateway, root):
    d = Path(self.out(root, 'rev-parse', '--git-dir').strip())
    return d if d.is_absolute() else root/d

@patch
def _journal_path(self:GitGateway, root):
    return self._git_dir(root)/JOURNAL_NAME

@patch
def journal(self:GitGateway, root):
    "Safepoints for this repository, newest first."
    try: raw = json.loads(self._journal_path(root).read_text(encoding='utf-8'))
    except (OSError, ValueError): return []
    return [Safepoint(**row) for row in raw if isinstance(row, dict)][:JOURNAL_KEEP]

@patch
def _record(self:GitGateway, root, point):
    rows = [asdict(point), *(asdict(p) for p in self.journal(root))][:JOURNAL_KEEP]
    path = self._journal_path(root)
    try:
        tmp = path.with_suffix('.tmp')
        tmp.write_text(json.dumps(rows, indent=1), encoding='utf-8')
        tmp.replace(path)
    except OSError: pass

In [ ]:
gw, r = GitGateway(), mkrepo()
test_eq(gw._git_dir(r), r/'.git')
test_eq(gw._journal_path(r), r/'.git'/JOURNAL_NAME)
test_eq(gw.journal(r), [])                              # no file yet

for i in range(3): gw._record(r, Safepoint(f't{i}', 'op', 'a'*40, 'main', '', 0, i))
test_eq([p.token for p in gw.journal(r)], ['t2', 't1', 't0'])    # newest first

for i in range(JOURNAL_KEEP + 5): gw._record(r, Safepoint(f'x{i}', 'op', 'a'*40, 'main', '', 0, i))
test_eq(len(gw.journal(r)), JOURNAL_KEEP)               # capped, oldest dropped

gw._journal_path(r).write_text('not json')
test_eq(gw.journal(r), [])                              # unreadable reads as empty, never raises
gw._journal_path(r).write_text('[1, 2, {"token": "ok", "op": "o", "head": "h", "branch": "b", '
                               '"stash": "", "dirty": 0, "created_at": 0}]')
test_eq([p.token for p in gw.journal(r)], ['ok'])       # non-object rows are skipped

## Creating a safepoint

`safepoint` uses `git stash create` to capture uncommitted work without cleaning the working tree. `transaction` holds the write lock for an operation and creates this safepoint before yielding.

In [ ]:
#| export
@patch
def safepoint(self:GitGateway, root, op):
    "Where the repository is, without touching it. None when no snapshot can be taken."
    head = self.run(root, 'rev-parse', 'HEAD', check=False)
    head = head.stdout.strip() if not head.returncode else ''
    if not head: return None
    branch = self.out(root, 'branch', '--show-current').strip()
    created = self.run(root, 'stash', 'create', check=False, kind='write')
    stash = created.stdout.strip() if not created.returncode else ''
    dirty = len([l for l in self.out(root, 'status', '--porcelain=v1').splitlines() if l])
    token = f'{int(time.time() * 1000):x}'
    if stash:
        self.run(root, 'update-ref', f'{SAFEPOINT_REF}/{token}/stash', stash, check=False)
    self.run(root, 'update-ref', f'{SAFEPOINT_REF}/{token}/head', head, check=False)
    point = Safepoint(token=token, op=str(op), head=head, branch=branch, stash=stash,
        dirty=dirty, created_at=int(time.time()))
    self._record(root, point)
    return point

@patch
@contextmanager
def transaction(self:GitGateway, root, op, snapshot=True):
    "Hold the write lock for a whole operation, with a way back recorded first."
    with self._lock(root).write():
        yield self.safepoint(root, op) if snapshot else None

In [ ]:
gw, r = GitGateway(), mkrepo()
base = sh(r, 'rev-parse', 'HEAD')
write(r, 'readme.md', 'dirty\n')
p = gw.safepoint(r, 'demo')
test_eq((p.op, p.branch, p.dirty, p.head), ('demo', 'main', 1, base))
test_eq((r/'readme.md').read_text(), 'dirty\n')         # snapshotting does not clean the tree
assert p.stash
test_eq(sh(r, 'rev-parse', f'{SAFEPOINT_REF}/{p.token}/head'), base)
test_eq([x.token for x in gw.journal(r)], [p.token])    # ...and it is journalled

with gw.transaction(r, 'held') as point: test_eq(point.op, 'held')
with gw.transaction(r, 'held', snapshot=False) as point: test_eq(point, None)

## Restoring a safepoint

`undo` first aborts an active merge, rebase, cherry-pick, or revert. It then restores the recorded branch and commit before applying the saved uncommitted work.

If the saved work cannot be applied, `undo` leaves it reachable as a commit and reports the recovery command. It never discards that work.

`gateway()` returns the process-wide `GitGateway` instance.

In [ ]:
#| export
@patch
def undo(self:GitGateway, root, token=''):
    "Restore a recorded safepoint and its uncommitted work."
    points = self.journal(root)
    if not points:
        raise GitError('nothing to undo -- no safepoint has been recorded for this repository')
    point = first(p for p in points if p.token == token) if token else points[0]
    if point is None: raise GitError(f'no safepoint {token} in this repository')
    with self._lock(root).write():
        active = self.active_operation(root)
        if active: self.run(root, active, '--abort', check=False)
        if point.branch and self.out(root, 'branch', '--show-current').strip() != point.branch:
            self.run(root, 'checkout', point.branch)
        self.run(root, 'reset', '--hard', point.head)
        restored = 0
        if point.stash:
            applied = self.run(root, 'stash', 'apply', '--index', point.stash, check=False)
            if applied.returncode:
                applied = self.run(root, 'stash', 'apply', point.stash, check=False)
            if applied.returncode:
                raise GitError(
                    f'HEAD is back at {point.head[:9]}, but the uncommitted work would not '
                    f'reapply cleanly. It is kept as commit {point.stash[:9]} -- recover it '
                    f'with `git stash apply {point.stash}`.\n\n{applied.stderr.strip()}')
            restored = point.dirty
    return {'token': point.token, 'op': point.op, 'head': point.head,
        'branch': point.branch, 'restored': restored, 'describe': point.describe()}

@patch
def active_operation(self:GitGateway, root):
    "Return the active Git operation name, or an empty string."
    git_dir = self._git_dir(root)
    for name, marker in (('rebase', 'rebase-merge'), ('rebase', 'rebase-apply'),
        ('merge', 'MERGE_HEAD'), ('cherry-pick', 'CHERRY_PICK_HEAD'),
        ('revert', 'REVERT_HEAD')):
        if (git_dir/marker).exists(): return name
    return ''

_GATEWAY = GitGateway()

def gateway():
    "Return the process-wide `GitGateway`."
    return _GATEWAY

In [ ]:
gw, r = GitGateway(), mkrepo()
test_eq(gateway() is gateway(), True)
base = sh(r, 'rev-parse', 'HEAD')
write(r, 'readme.md', 'dirty\n')
p = gw.safepoint(r, 'demo')
commit(r, 'second', readme__md='second\n')

out = gw.undo(r)
test_eq((out['token'], out['restored'], out['op']), (p.token, 1, 'demo'))
test_eq(sh(r, 'rev-parse', 'HEAD'), base)               # back to where the safepoint said
test_eq((r/'readme.md').read_text(), 'dirty\n')         # ...with the uncommitted work reapplied

test_eq(gw.active_operation(r), '')
test_fail(lambda: gw.undo(r, 'nosuch'), contains='no safepoint nosuch')
test_fail(lambda: gw.undo(mkrepo()), contains='nothing to undo')

## Finding and cloning repositories

`repo_root` finds the worktree that contains a path. `url_name` derives a folder name from a Git URL. `clone_target` accepts only a new direct child of an existing directory. It rejects option-like URLs and existing targets. `clone` performs the clone and returns its root.

In [ ]:
#| export
def _run(cwd, *args, check=True, input=None, timeout=30, kind=None):
    "One Git command, through the gateway that serialises it against the repository's others."
    return gateway().run(cwd, *args, check=check, input=input, timeout=timeout, kind=kind)

def repo_root(path):
    "The containing worktree root, or `None` when path is not in a repository."
    p = Path(path).expanduser().resolve()
    if p.is_file(): p = p.parent
    try:
        r = _run(p, 'rev-parse', '--show-toplevel', check=False)
        return Path(r.stdout.strip()).resolve() if r.returncode == 0 and r.stdout.strip() else None
    except (OSError, GitError): return None

def url_name(url):
    "The folder name a clone of `url` lands in: its last path segment, without `.git`."
    return re.sub(r'\.git$', '', str(url or '').rstrip('/').rpartition('/')[2])

def clone_target(url, parent, name=''):
    "Where a clone of `url` would land, refusing anything but a free folder inside `parent`."
    url = str(url or '').strip()
    if not url or url.startswith('-'): raise GitError(f'not a repository URL: {url or "(empty)"}')
    parent = Path(parent).expanduser().resolve()
    if not parent.is_dir(): raise GitError(f'not a directory: {parent}')
    name = str(name or '').strip() or url_name(url)
    if not name or '/' in name or name in ('.', '..'):
        raise GitError(f'cannot work out a folder name for {url}')
    target = parent/name
    if target.exists(): raise GitError(f'{name} already exists in {parent.name}')
    return target

def clone(url, parent, name=''):
    "Clone `url` into `parent`, returning the new worktree root."
    target = clone_target(url, parent, name)
    _run(target.parent, 'clone', '--', str(url).strip(), target.name, timeout=600)
    return target

In [ ]:
r = mkrepo()
write(r, 'src/app.py', 'x = 1\n')
test_eq(repo_root(r), r.resolve())
test_eq(repo_root(r/'src'), r.resolve())                # a folder inside
test_eq(repo_root(r/'src'/'app.py'), r.resolve())       # ...and a file inside
test_eq(repo_root(Path(tempfile.mkdtemp())), None)      # not a repository at all

test_eq(url_name('https://github.com/answerdotai/fastcore.git'), 'fastcore')
test_eq(url_name('git@github.com:answerdotai/fastcore.git'), 'fastcore')
test_eq(url_name('https://github.com/answerdotai/fastcore/'), 'fastcore')
test_eq(url_name(''), '')

parent = Path(tempfile.mkdtemp()).resolve(); _tmp.append(str(parent))
test_eq(clone_target('https://h/o/thing.git', parent), parent/'thing')
test_eq(clone_target('https://h/o/thing.git', parent, 'other'), parent/'other')
test_fail(lambda: clone_target('', parent), contains='not a repository URL')
test_fail(lambda: clone_target('--upload-pack=evil', parent), contains='not a repository URL')
test_fail(lambda: clone_target('https://h/o/t.git', parent/'nope'), contains='not a directory')
test_fail(lambda: clone_target('https://h/o/t.git', parent, 'a/b'), contains='folder name')
(parent/'taken').mkdir()
test_fail(lambda: clone_target('https://h/o/taken.git', parent), contains='already exists')

made = clone(str(r), parent, 'copy')                    # a local path is a URL git understands
test_eq(made, parent/'copy')
test_eq(repo_root(made), made.resolve())

## Parsing conflict markers

`_conflict_blocks` returns spans that contain a complete start, separator, and end marker. Incomplete marker sequences remain ordinary text.

In [ ]:
#| export
def _conflict_blocks(text):
    "Marker spans with their ours/theirs payload. Malformed markers remain ordinary text."
    out, pos = [], 0
    while True:
        start = text.find('<<<<<<< ', pos)
        if start < 0: break
        middle = text.find('\n=======\n', start)
        end = text.find('\n>>>>>>> ', middle + 8) if middle >= 0 else -1
        finish = text.find('\n', end + 1) if end >= 0 else -1
        if middle < 0 or end < 0: pos = start + 7; continue
        finish = len(text) if finish < 0 else finish + 1
        ours = text[text.find('\n', start) + 1:middle + 1]
        theirs = text[middle + 9:end + 1]
        out.append({'start': start, 'end': finish, 'ours': ours, 'theirs': theirs}); pos = finish
    return out

In [ ]:
block, = _conflict_blocks('a\n<<<<<<< HEAD\nx\n=======\ny\n>>>>>>> o\nb\n')
test_eq((block['ours'], block['theirs']), ('x\n', 'y\n'))
test_eq(block['start'], 2)

test_eq(_conflict_blocks('nothing here\n'), [])
test_eq(_conflict_blocks('<<<<<<< HEAD\nx\nno middle\n'), [])       # malformed stays text
test_eq(_conflict_blocks('<<<<<<< HEAD\nx\n=======\ny\n'), [])      # no closing marker
test_eq(len(_conflict_blocks('<<<<<<< a\n1\n=======\n2\n>>>>>>> b\n'
                             '<<<<<<< a\n3\n=======\n4\n>>>>>>> b\n')), 2)
test_eq(_conflict_blocks('<<<<<<< a\n\n=======\n\n>>>>>>> b\n')[0]['ours'], '\n')

## Short-lived status caching

A file-tree refresh can ask for the same status more than once. `_cached` shares that result for `_CACHE_TTL` seconds. Mutations invalidate the entry, and direct gutter reads remain uncached so a saved file appears immediately.

In [ ]:
#| export
_CACHE, _CACHE_LOCK = {}, threading.RLock()

_CACHE_TTL = .45

def _cached(root, key, make):
    cache_key = (str(root), key)
    now = time.monotonic()
    with _CACHE_LOCK:
        hit = _CACHE.get(cache_key)
        if hit and now - hit[0] < _CACHE_TTL: return hit[1]
    value = make()
    with _CACHE_LOCK: _CACHE[cache_key] = (time.monotonic(), value)
    return value

def invalidate(root):
    "Drop every cached answer for `root`, after something wrote to it."
    root = str(root)
    with _CACHE_LOCK:
        for key in [k for k in _CACHE if k[0] == root]: _CACHE.pop(key, None)

In [ ]:
calls = []
def make(): calls.append(1); return len(calls)
root = '/some/root'
test_eq(_cached(root, 'k', make), 1)
test_eq(_cached(root, 'k', make), 1)                    # inside the ttl
test_eq(_cached('/other', 'k', make), 2)                # keyed by root as well as key
invalidate(root)
test_eq(_cached(root, 'k', make), 3)                    # a mutation drops what it invalidated
test_eq(_cached('/other', 'k', make), 2)                # ...and only that root
invalidate('/other')

## Small readings

The parsers the status and branch readers need, and the two spellings of a number that the prose
below uses.

In [ ]:
#| export
def _remote_web_url(raw):
    "Convert a supported Git transport URL to an HTTP repository URL."
    raw = (raw or '').strip()
    if not raw: return ''
    if re.match(r'^[^/@:]+@[^:]+:.+', raw):
        user_host, path = raw.split(':', 1)
        raw = f'https://{user_host.split("@", 1)[1]}/{path}'
    elif raw.startswith('ssh://'):
        parsed = urlparse(raw)
        raw = f'https://{parsed.hostname or ""}{parsed.path}'
    elif raw.startswith('git://'): raw = 'https://' + raw[6:]
    if raw.startswith(('http://', 'https://')): return raw.removesuffix('.git').rstrip('/')
    return ''

def _track_counts(track):
    "Parse Git's upstream tracking text into `(ahead, behind)`."
    counts = dict(re.findall(r'(ahead|behind) (\d+)', track or ''))
    return int(counts.get('ahead', 0)), int(counts.get('behind', 0))

def plural(n, word):
    "Return a count with a singular or plural noun."
    return f'{n} {word}' + ('' if n == 1 else 's')

def shorten(text, limit):
    "Collapse whitespace and truncate `text` to `limit` characters."
    text = ' '.join(str(text or '').split())
    return text if len(text) <= limit else text[:limit - 1] + '…'

def unborn(oid):
    "Return whether `oid` is Git's all-zero object ID."
    return set(str(oid)) == {'0'}

def _fields(
    text,
    n,
):
    "Split a NUL-separated record into exactly `n` fields."
    return (text.split('\0') + [''] * n)[:n]

In [ ]:
test_eq(_remote_web_url('git@github.com:o/r.git'), 'https://github.com/o/r')
test_eq(_remote_web_url('ssh://git@github.com/o/r.git'), 'https://github.com/o/r')
test_eq(_remote_web_url('git://github.com/o/r.git'), 'https://github.com/o/r')
test_eq(_remote_web_url('https://github.com/o/r/'), 'https://github.com/o/r')
test_eq(_remote_web_url('/local/path'), '')             # nothing a browser could open
test_eq(_remote_web_url(''), '')

test_eq(_track_counts('[ahead 2, behind 1]'), (2, 1))
test_eq(_track_counts('[ahead 3]'), (3, 0))
test_eq(_track_counts(''), (0, 0))
test_eq(_track_counts('[gone]'), (0, 0))

test_eq(plural(1, 'file'), '1 file')
test_eq(plural(0, 'file'), '0 files')

test_eq(shorten('a  b   c', 4), 'a b…')
test_eq(shorten('short', 40), 'short')
test_eq(shorten(None, 4), '')

assert unborn('0' * 40)
assert not unborn('0' * 39 + '1')

test_eq(_fields('a\0b', 4), ['a', 'b', '', ''])         # padded to the width asked for
test_eq(_fields('a\0b\0c', 2), ['a', 'b'])              # ...and truncated to it

## Reporting outcomes

Every guarded mutation returns the same dictionary. `_summarise` turns it into one action sentence. Conflicts take priority, followed by an active operation and staged work that still needs a commit.

`_explain_filters` reports missing or unconfigured Git content filters.

In [ ]:
#| export
def _summarise(out):
    "Summarize a mutation outcome in one sentence."
    op, staged, conflicted = out['op'], len(out['staged']), len(out['conflicted'])
    if conflicted:
        return f'{op} stopped at {plural(conflicted, "conflicted file")} -- resolve them to continue'
    if out['operation']['active']:
        return f'{out["operation"]["active"]} is in progress -- continue, skip, or abort it'
    if staged and not out['moved']:
        return f'{op} staged {plural(staged, "file")} without committing -- commit to finish'
    if out['moved']:
        return f'{op} moved this branch to {out["head"]}'
    return out['message'].splitlines()[0] if out['message'] else f'{op} changed nothing'

def _explain_filters(broken):
    "Explain missing and unconfigured Git content filters."
    missing = [r for r in broken if r['configured']]
    unconfigured = [r for r in broken if not r['configured']]
    lines = []
    if missing:
        which = ', '.join(f'{r["name"]} (`{shorten(r["clean"] or r["smudge"], 60)}`)' for r in missing)
        lines.append(
            f'This repository cleans files through {which}, and that command cannot be found in '
            f'the environment Leela runs Git in. Git does not treat a failing filter as an error, '
            f'so affected files -- notebooks, usually -- will keep reappearing as modified however '
            f'often you stage them. Installing it into this project\'s virtualenv fixes it.')
    if unconfigured:
        lines.append(
            f'{", ".join(r["name"] for r in unconfigured)} is named in .gitattributes but '
            f'configured nowhere, so Git is passing that content through unchanged.')
    return '\n\n'.join(lines)

In [ ]:
base = {'op': 'merge', 'staged': [], 'conflicted': [], 'moved': False, 'head': 'abc1234',
        'message': '', 'operation': {'active': ''}}
test_eq(_summarise(base), 'merge changed nothing')
test_eq(_summarise(base | {'moved': True}), 'merge moved this branch to abc1234')
test_eq(_summarise(base | {'staged': ['a']}), 'merge staged 1 file without committing -- commit to finish')
test_eq(_summarise(base | {'operation': {'active': 'rebase'}}),
        'rebase is in progress -- continue, skip, or abort it')
test_eq(_summarise(base | {'conflicted': ['a', 'b']}),
        'merge stopped at 2 conflicted files -- resolve them to continue')
test_eq(_summarise(base | {'message': 'Already up to date.\nmore'}), 'Already up to date.')
# a conflict outranks everything else, because it is what the person has to deal with
test_eq(_summarise(base | {'conflicted': ['a'], 'staged': ['a'], 'moved': True}),
        'merge stopped at 1 conflicted file -- resolve them to continue')

assert 'cannot be found' in _explain_filters([{'name': 'nbstripout', 'configured': True,
                                               'clean': 'nbstripout', 'smudge': ''}])
assert 'configured nowhere' in _explain_filters([{'name': 'lfs', 'configured': False,
                                                  'clean': '', 'smudge': ''}])
test_eq(_explain_filters([]), '')

## Reading a diff

Git's porcelain diff output is NUL-separated and has a three-field rename form that the ordinary
two-field parse silently misreads. Both parsers below handle it, and both keep `None` counts for a
binary file rather than reporting it as zero lines changed.

In [ ]:
#| export
def _unified_arg(context):
    "`--unified=n`, clamped to what a diff pane can usefully show."
    return f'--unified={max(0, min(int(context), 20))}'

def _unified(before, after, from_label, to_label):
    "One unified diff of two texts, as `git diff` would print it."
    return ''.join(difflib.unified_diff(before.splitlines(True), after.splitlines(True),
        fromfile=from_label, tofile=to_label, n=3))

def _summarise_diff(rows):
    "Totals across the per-file records of one comparison."
    return {'files': len(rows), 'additions': sum(x['additions'] or 0 for x in rows),
        'deletions': sum(x['deletions'] or 0 for x in rows),
        'binary': sum(bool(x['binary']) for x in rows)}

def _diff_status(raw):
    "Parse `git diff --name-status -z` into rename-aware records."
    fields, rows, i = raw.split('\0'), [], 0
    while i < len(fields) and fields[i]:
        status, i = fields[i], i + 1
        path = fields[i] if i < len(fields) else ''
        i += 1
        old_path = ''
        if status[:1] in {'R', 'C'}:
            old_path, path = path, fields[i] if i < len(fields) else path
            i += 1
        rows.append({'path': path, 'old_path': old_path, 'status': status[:1],
            'similarity': int(status[1:] or 0)})
    return rows

def _diff_numstat(raw):
    "Parse `git diff --numstat -z` including its three-field rename form."
    fields, stats, i = raw.split('\0'), {}, 0
    while i < len(fields) and fields[i]:
        head, i = fields[i], i + 1
        bits = head.split('\t', 2)
        if len(bits) != 3: continue
        added, deleted, path = bits
        if not path and i + 1 < len(fields):
            _old, path, i = fields[i], fields[i + 1], i + 2
        stats[path] = {'additions': None if added == '-' else int(added or 0),
            'deletions': None if deleted == '-' else int(deleted or 0),
            'binary': added == '-' or deleted == '-'}
    return stats

In [ ]:
test_eq(_unified_arg(3), '--unified=3')
test_eq(_unified_arg(-1), '--unified=0')                # clamped both ways
test_eq(_unified_arg(999), '--unified=20')

d = _unified('a\n', 'b\n', 'a/x', 'b/x')
assert d.startswith('--- a/x') and '-a' in d and '+b' in d
test_eq(_unified('same\n', 'same\n', 'a', 'b'), '')

rows = _diff_status('M\0a.py\0R090\0old.py\0new.py\0A\0c.py\0')
test_eq([r['path'] for r in rows], ['a.py', 'new.py', 'c.py'])
test_eq(rows[1]['old_path'], 'old.py')
test_eq((rows[1]['status'], rows[1]['similarity']), ('R', 90))
test_eq(rows[0]['similarity'], 0)
test_eq(_diff_status(''), [])

stats = _diff_numstat('1\t2\ta.py\0-\t-\tbin.png\0')
test_eq(stats['a.py'], {'additions': 1, 'deletions': 2, 'binary': False})
test_eq(stats['bin.png'], {'additions': None, 'deletions': None, 'binary': True})
test_eq(_diff_numstat('3\t4\t\0old.py\0new.py\0')['new.py']['additions'], 3)   # the rename form
test_eq(_diff_numstat(''), {})

test_eq(_summarise_diff([{'additions': 1, 'deletions': 2, 'binary': False},
                         {'additions': None, 'deletions': None, 'binary': True}]),
        {'files': 2, 'additions': 1, 'deletions': 2, 'binary': 1})
test_eq(_summarise_diff([]), {'files': 0, 'additions': 0, 'deletions': 0, 'binary': 0})

## Opening a repository

`GitRepo` stores a worktree root and exposes operations for that repository. `GitRepo.at` accepts any path inside the worktree.

`_ask` returns stripped output or an empty string for a failed optional read. `_mutate` runs a command and invalidates cached state. `_gitdir` locates Git's private directory. `_version` reads a file from the working tree, index, or `HEAD`.

In [ ]:
#| export
@dataclass
class GitRepo:
    root: Path
    AUTOSTASH_REF = 'refs/leela/autostash'   # kept, like `SAFEPOINT_REF`: it names refs already on disk
    _WOULD_CLOBBER = ('would be overwritten', 'local changes')

@patch(cls_method=True)
def at(cls:GitRepo, path):
    root = repo_root(path)
    if root is None: raise GitError(f'{Path(path).name or path} is not inside a Git repository')
    return cls(root)

@patch
def run(self:GitRepo, *args, **kwargs):
    return _run(self.root, *args, **kwargs).stdout

@patch
def _ask(self:GitRepo, *args):
    "Stripped stdout of a command allowed to fail, or `''`."
    r = _run(self.root, *args, check=False)
    return r.stdout.strip() if not r.returncode else ''

@patch
def _mutate(self:GitRepo, *args, **kwargs):
    try:
        return self.run(*args, **kwargs)
    finally:
        invalidate(self.root)

@patch
def _gitdir(self:GitRepo):
    git_dir = Path(self.run('rev-parse', '--git-dir').strip())
    return git_dir if git_dir.is_absolute() else self.root/git_dir

@patch
def _version(self:GitRepo, path, where):
    if where == 'worktree':
        try: return (self.root/path).read_text(errors='replace')
        except OSError: return ''
    spec = f':{path}' if where == 'index' else f'HEAD:{path}'
    r = _run(self.root, 'show', spec, check=False)
    return r.stdout if r.returncode == 0 else ''

In [ ]:
r = mkrepo()
write(r, 'src/app.py', 'x = 1\n')
test_eq(GitRepo.at(r).root, r.resolve())
test_eq(GitRepo.at(r/'src'/'app.py').root, r.resolve())
test_fail(lambda: GitRepo.at(Path(tempfile.mkdtemp())), contains='not inside a Git repository')

repo = GitRepo.at(r)
test_eq(repo.run('branch', '--show-current').strip(), 'main')
test_eq(repo._ask('branch', '--show-current'), 'main')
test_eq(repo._ask('rev-parse', 'nope'), '')             # a read that fails answers with nothing
test_fail(lambda: repo.run('rev-parse', 'nope'))        # ...where `run` still raises
test_eq(repo._mutate('add', '-A'), '')
test_eq(repo._ask('diff', '--cached', '--name-only'), 'src/app.py')

test_eq(repo._gitdir(), r.resolve()/'.git')
test_eq(repo._version('readme.md', 'worktree'), 'hello\n')
test_eq(repo._version('readme.md', 'head'), 'hello\n')
test_eq(repo._version('src/app.py', 'head'), '')        # not committed yet
test_eq(repo._version('missing.txt', 'worktree'), '')

## Where the repository is

The operation in progress, the remotes and their web URLs, and what the current branch tracks. Each
is a small read on its own so that `info` can assemble them and a caller can ask for one.

In [ ]:
#| export
@patch
def _operation(self:GitRepo):
    git_dir = self._gitdir()
    checks = (
        ('rebase', git_dir/'rebase-merge'), ('rebase', git_dir/'rebase-apply'),
        ('merge', git_dir/'MERGE_HEAD'), ('cherry-pick', git_dir/'CHERRY_PICK_HEAD'),
        ('revert', git_dir/'REVERT_HEAD'), ('bisect', git_dir/'BISECT_LOG'),
    )
    active = first(name for name, path in checks if path.exists()) or ''
    return {'active': active, 'can_continue': active in {'rebase', 'cherry-pick', 'revert'},
        'can_skip': active in {'rebase', 'cherry-pick'}, 'can_abort': bool(active)}

@patch
def _remote_names(self:GitRepo):
    "The configured remotes, in Git's own order."
    return [n for n in self.run('remote').splitlines() if n]

@patch
def _remotes(self:GitRepo):
    rows = []
    for name in self._remote_names():
        fetch = self._ask('remote', 'get-url', name)
        rows.append({'name': name, 'fetch': fetch, 'push': self._ask('remote', 'get-url', '--push', name),
            'web_url': _remote_web_url(fetch)})
    return rows

@patch
def _tracking(self:GitRepo):
    "The upstream this branch tracks, and how far each side has run ahead of the other."
    upstream = self._ask('rev-parse', '--abbrev-ref', '@{upstream}')
    if not upstream: return '', 0, 0
    counts = self.run('rev-list', '--left-right', '--count', f'{upstream}...HEAD').split()
    behind, ahead = map(int, counts) if len(counts) == 2 else (0, 0)
    return upstream, ahead, behind

In [ ]:
r = GitRepo.at(mkrepo())
test_eq(r._operation()['active'], '')
test_eq(r._remote_names(), [])
test_eq(r._remotes(), [])
test_eq(r._tracking(), ('', 0, 0))                      # upstream, ahead, behind
test_eq(r._operation()['can_abort'], False)

bare = mkbare()
sh(r.root, 'remote', 'add', 'origin', str(bare))
sh(r.root, 'remote', 'set-url', 'origin', 'git@github.com:o/r.git')
test_eq(r._remote_names(), ['origin'])
test_eq(r._remotes()[0]['name'], 'origin')
test_eq(r._remotes()[0]['web_url'], 'https://github.com/o/r')
sh(r.root, 'remote', 'set-url', 'origin', str(bare))
sh(r.root, 'push', '-u', 'origin', 'main')
commit(r.root, 'ahead', readme__md='ahead\n')
test_eq(r._tracking(), ('origin/main', 1, 0))

## Reading changes

`git status --porcelain=v1 -z` supplies file state. Two `--numstat` calls add staged and unstaged line counts. Untracked files are read directly because Git has no earlier version to compare.

`changes` always reads fresh state. `decorations` caches the same shape for one file-tree refresh.

In [ ]:
#| export
@patch
def _changes_uncached(self:GitRepo):
    raw = _run(self.root, 'status', '--porcelain=v1', '-z').stdout
    fields, out, i = raw.split('\0'), [], 0
    while i < len(fields) and fields[i]:
        row = fields[i]
        i += 1
        xy, path = row[:2], row[3:]
        old = None
        if ('R' in xy or 'C' in xy) and i < len(fields):
            old, i = fields[i], i + 1
        out.append({
            'path': path, 'old_path': old, 'index': xy[0], 'worktree': xy[1],
            'staged': xy[0] not in (' ', '?'),
            'unstaged': xy[1] != ' ' or xy == '??', 'untracked': xy == '??',
            'conflicted': xy in ('DD', 'AU', 'UD', 'UA', 'DU', 'AA', 'UU'),
        })
    unstaged = _diff_numstat(_run(self.root, 'diff', '--numstat', '-z', check=False).stdout)
    staged = _diff_numstat(_run(self.root, 'diff', '--cached', '--numstat', '-z', check=False).stdout)
    for row in out:
        row['staged_stat'] = staged.get(row['path'])
        row['unstaged_stat'] = unstaged.get(row['path'])
        if row['untracked']:
            try:
                data = (self.root/row['path']).read_bytes()
                binary = b'\0' in data[:8192]
                row['unstaged_stat'] = {'additions': None if binary else len(data.decode('utf-8', 'replace').splitlines()),
                    'deletions': None if binary else 0, 'binary': binary}
            except OSError: pass
    return out

@patch
def changes(self:GitRepo):
    "Current status. Uncached: the gutter asks right after a save and must see it."
    return self._changes_uncached()

@patch
def decorations(self:GitRepo):
    "The same status for the file tree, which asks once per open folder in one refresh."
    return _cached(self.root, 'changes', self._changes_uncached)

In [ ]:
r = GitRepo.at(mkrepo())
test_eq(r.changes(), [])

write(r.root, 'new.txt', 'a\nb\n')
write(r.root, 'readme.md', 'changed\n')
rows = {c['path']: c for c in r.changes()}
test_eq(sorted(rows), ['new.txt', 'readme.md'])
test_eq((rows['new.txt']['untracked'], rows['new.txt']['staged']), (True, False))
test_eq(rows['new.txt']['unstaged_stat'], {'additions': 2, 'deletions': 0, 'binary': False})
test_eq((rows['readme.md']['untracked'], rows['readme.md']['unstaged']), (False, True))
test_eq(rows['readme.md']['unstaged_stat']['additions'], 1)
test_eq(rows['readme.md']['staged_stat'], None)

sh(r.root, 'add', 'readme.md')                          # staged outside the module under test
rows = {c['path']: c for c in r.changes()}
test_eq((rows['readme.md']['staged'], rows['readme.md']['index']), (True, 'M'))
test_eq(rows['readme.md']['staged_stat']['additions'], 1)

write(r.root, 'bin.dat', 'a\0b')
test_eq(first(c for c in r.changes() if c['path'] == 'bin.dat')['unstaged_stat']['binary'], True)
test_eq(len(r.decorations()), len(r.changes()))         # the same answer, one of them cached

## What is in the way

Uncommitted work is what stops a merge or a rebase starting. `_in_the_way` is that work in the shape
`divergence` reports it, so the recommendation can take it into account.

In [ ]:
#| export
@patch
def _blocking_changes(self:GitRepo):
    "Tracked files with uncommitted work: what actually stops a merge or a rebase."
    return [c for c in self.changes() if not c.get('untracked')]

@patch
def _in_the_way(self:GitRepo):
    "How much uncommitted work a preview has to warn about, in the shape both previews report."
    blocking = self._blocking_changes()
    return {'clean': not blocking, 'dirty': [c['path'] for c in blocking][:20],
        'untracked': sum(1 for c in self.changes() if c.get('untracked'))}

In [ ]:
r = GitRepo.at(mkrepo({'a.txt': 'a\n'}))
test_eq(r._blocking_changes(), [])
test_eq(r._in_the_way()['clean'], True)

write(r.root, 'a.txt', 'edited\n')
test_eq([c['path'] for c in r._blocking_changes()], ['a.txt'])
test_eq(r._in_the_way()['clean'], False)
test_eq(r._in_the_way()['dirty'], ['a.txt'])

sh(r.root, 'checkout', '--', 'a.txt')
write(r.root, 'untracked.txt', 'new\n')
test_eq(r._in_the_way()['clean'], True)                 # untracked work is not in the way

## Preserving uncommitted work

Some operations require a clean working tree. `_set_aside` stores tracked changes under `AUTOSTASH_REF` and resets the tree. `_bring_back` reapplies them after the operation.

The ref keeps the snapshot reachable if it cannot be reapplied. In that case `_bring_back` leaves the new result intact and returns a recovery command.

In [ ]:
#| export
@patch
def _set_aside(self:GitRepo, op):
    "Snapshot the dirty tree and clear it, returning the commit. Or `''` if it was clean."
    if not self._blocking_changes(): return ''
    created = self._ask('stash', 'create')
    if not created: return ''
    _run(self.root, 'update-ref', f'{self.AUTOSTASH_REF}/{op}', created, check=False)
    _run(self.root, 'reset', '--hard', check=False)
    return created

@patch
def _bring_back(self:GitRepo, oid):
    "Reapply a set-aside tree. Returns a note when it would not go back on cleanly."
    applied = _run(self.root, 'stash', 'apply', '--index', oid, check=False)
    if applied.returncode:
        _run(self.root, 'reset', '--hard', check=False)
        applied = _run(self.root, 'stash', 'apply', oid, check=False)
    if applied.returncode:
        _run(self.root, 'reset', '--hard', check=False)
        return (f'Your uncommitted work would not go back on top of this result, so it has '
            f'been left as commit {oid[:9]} -- recover it with `git stash apply {oid}`.')
    return ''

In [ ]:
r = GitRepo.at(mkrepo())
test_eq(r._set_aside('demo'), '')                       # a clean tree has nothing to set aside

write(r.root, 'readme.md', 'mine\n')
oid = r._set_aside('demo')
assert oid
test_eq((r.root/'readme.md').read_text(), 'hello\n')    # the tree is clean again
test_eq(sh(r.root, 'rev-parse', f'{r.AUTOSTASH_REF}/demo'), oid)

test_eq(r._bring_back(oid), '')
test_eq((r.root/'readme.md').read_text(), 'mine\n')     # ...and the work comes back

# Work that cannot go back on top is kept as a commit, and the note says how to reach it.
r2 = GitRepo.at(mkrepo())
write(r2.root, 'readme.md', 'mine\n')
kept = r2._set_aside('demo')
commit(r2.root, 'moved on', readme__md='theirs\n')
note = r2._bring_back(kept)
assert 'git stash apply' in note and kept[:9] in note
test_eq((r2.root/'readme.md').read_text(), 'theirs\n')  # ...and the tree is left consistent

## Mutation results

A merge conflict is a valid result, not a command failure. `_attempt` returns conflicts while raising `GitError` for other failures. `_guarded` adds a safepoint and restores set-aside work. `_outcome` builds the common response returned by guarded mutations.

In [ ]:
#| export
@patch
def _attempt(self:GitRepo, *args, **kwargs):
    "Run a mutation whose conflicts are an outcome rather than a failure."
    kwargs['check'] = False
    try:
        p = _run(self.root, *args, **kwargs)
    finally:
        invalidate(self.root)
    text = '\n'.join(x for x in ((p.stdout or '').strip(), (p.stderr or '').strip()) if x)
    if not p.returncode:
        return text
    if self._operation()['active'] or any(c['conflicted'] for c in self._changes_uncached()):
        return text
    raise GitError(text or f'git {args[0] if args else "command"} exited {p.returncode}')

@patch
def _guarded(self:GitRepo, op, call, autostash=False):
    "One risky mutation: a way back recorded first, and a report of what it left behind."
    with gateway().transaction(self.root, op) as point:
        aside = self._set_aside(op) if autostash else ''
        failure, message = None, ''
        try:
            message = call() or ''
        except GitError as e:
            failure = e
        finally:
            invalidate(self.root)
        note = self._bring_back(aside) if aside else ''
        invalidate(self.root)
        if failure is not None:
            if note:
                raise GitError(f'{failure}\n\n{note}') from failure
            raise failure
        return self._outcome(op, point, message, note, bool(aside))

@patch
def _outcome(self:GitRepo, op, point, message='', note='', set_aside=False):
    "What one mutation left the repository as. The response every mutation returns."
    changes = self._changes_uncached()
    after = self._ask('rev-parse', '--short', 'HEAD')
    out = {
        'op': op, 'message': (message or '').strip(), 'note': note,
        'head': after, 'head_before': point.head[:9] if point else '',
        'moved': bool(point) and bool(after) and not point.head.startswith(after),
        'branch': self.run('branch', '--show-current').strip(),
        'staged': [c['path'] for c in changes if c['staged']],
        'conflicted': [c['path'] for c in changes if c['conflicted']],
        'operation': self._operation(),
        'kept_work': set_aside and not note,
        'undo': point.token if point else '', 'undoes': point.describe() if point else '',
    }
    out['summary'] = _summarise(out)
    return out

In [ ]:
r = GitRepo.at(mkrepo())
test_eq(r._attempt('status', '--porcelain=v1'), '')
test_fail(lambda: r._attempt('checkout', 'nope'), contains='did not match')

before = sh(r.root, 'rev-parse', 'HEAD')
out = r._guarded('demo', lambda: r._attempt('commit', '--allow-empty', '-m', 'empty'))
test_eq(out['op'], 'demo')
test_eq(out['branch'], 'main')
test_eq(out['head_before'], before[:9])
test_eq(out['moved'], True)
test_eq((out['staged'], out['conflicted']), ([], []))
test_eq(out['operation']['active'], '')
test_eq(out['summary'], f"demo moved this branch to {out['head']}")
assert out['undo'] and 'demo on main' in out['undoes']
test_eq(gateway().journal(r.root)[0].token, out['undo'])

# A failure inside the guard still raises, and a safepoint is recorded before it does.
test_fail(lambda: r._guarded('bad', lambda: r._attempt('checkout', 'nope')), contains='did not match')
test_eq(gateway().journal(r.root)[0].op, 'bad')

# Work set aside for a mutation that fails is still brought back.
write(r.root, 'readme.md', 'mine\n')
test_fail(lambda: r._guarded('bad', lambda: r._attempt('checkout', 'nope'), autostash=True))
test_eq((r.root/'readme.md').read_text(), 'mine\n')

## Undo, from the outside

`undo` and `safepoints` are the porcelain over the gateway's journal. `undo` with no token means
the most recent safepoint, which is what an undo button asks for.

In [ ]:
#| export
@patch
def undo(self:GitRepo, token=''):
    "Put this repository back where the named safepoint says it was."
    outcome = gateway().undo(self.root, token)
    invalidate(self.root)
    return outcome

@patch
def safepoints(self:GitRepo):
    "The recorded ways back for this repository, newest first."
    return [{'token': p.token, 'op': p.op, 'head': p.head[:9], 'branch': p.branch,
        'dirty': p.dirty, 'created_at': p.created_at, 'describe': p.describe()}
        for p in gateway().journal(self.root)]

In [ ]:
r = GitRepo.at(mkrepo())
test_eq(r.safepoints(), [])
base = sh(r.root, 'rev-parse', 'HEAD')

r._guarded('one', lambda: r._attempt('commit', '--allow-empty', '-m', 'a'))
r._guarded('two', lambda: r._attempt('commit', '--allow-empty', '-m', 'b'))
test_eq([p['op'] for p in r.safepoints()], ['two', 'one'])
test_eq(len(r.safepoints()[0]['head']), 9)              # shortened for display

test_eq(r.undo()['op'], 'two')                          # no token means the newest
test_eq(r.undo(r.safepoints()[-1]['token'])['op'], 'one')
test_eq(sh(r.root, 'rev-parse', 'HEAD'), base)
test_fail(lambda: r.undo('nosuch'), contains='no safepoint nosuch')

## Listing branches

One `for-each-ref` call returns each branch's commit, upstream, divergence, author, and date. Remote-tracking branches already claimed by a local branch are omitted to avoid duplicate entries.

In [ ]:
#| export
@patch
def _branches(self:GitRepo):
    "Local and unclaimed remote-tracking branches, newest commit first."
    fmt = '%(refname)%00%(refname:short)%00%(HEAD)%00%(upstream:short)%00' \
          '%(upstream:trackshort)%00%(objectname:short)%00%(subject)%00' \
          '%(committerdate:unix)%00%(committerdate:iso-strict)%00%(authorname)%00%(symref)' \
          '%00%(upstream:track)'
    raw = self.run('for-each-ref', f'--format={fmt}', '--sort=-committerdate',
        'refs/heads', 'refs/remotes')
    parsed, claimed = [], set()
    for line in raw.splitlines():
        (full, name, head, upstream, track, oid,
            subject, stamp, date, author, symref, drift) = _fields(line, 12)
        if full.startswith('refs/remotes/') and (symref or full.endswith('/HEAD')):
            continue
        remote = full.startswith('refs/remotes/')
        if not remote and upstream:
            claimed.add(upstream)
        ahead, behind = _track_counts(drift)
        parsed.append({
            'name': name, 'full_name': full, 'current': head == '*',
            'upstream': upstream, 'track': track, 'oid': oid, 'subject': subject,
            'remote': remote, 'timestamp': int(stamp or 0), 'date': date,
            'author': author, 'ahead': ahead, 'behind': behind, 'gone': 'gone' in drift,
        })
    merged = set(_run(self.root, 'for-each-ref', '--merged=HEAD', '--format=%(refname)',
        'refs/heads', 'refs/remotes', check=False).stdout.splitlines())
    rows = [r for r in parsed if not (r['remote'] and r['name'] in claimed)]
    newest = max([r['timestamp'] for r in rows], default=0)
    default = self._ask('symbolic-ref', '--quiet', '--short', 'refs/remotes/origin/HEAD')
    for row in rows:
        row['merged'] = row['full_name'] in merged
        row['latest'] = row['timestamp'] == newest
        row['default'] = row['name'] in {default, default.partition('/')[2]}
    return rows

In [ ]:
r = GitRepo.at(mkrepo())
rows = r._branches()
test_eq([b['name'] for b in rows], ['main'])
main, = rows
test_eq((main['current'], main['remote'], main['upstream']), (True, False, ''))
test_eq((main['ahead'], main['behind'], main['gone']), (0, 0, False))
test_eq((main['merged'], main['latest']), (True, True))
test_eq(main['subject'], 'initial')
test_eq(main['author'], 'Repo tests')

sh(r.root, 'branch', 'feature')
test_eq(sorted(b['name'] for b in r._branches()), ['feature', 'main'])

bare = mkbare()
sh(r.root, 'remote', 'add', 'origin', str(bare))
sh(r.root, 'push', '-u', 'origin', 'main')
commit(r.root, 'ahead one', readme__md='ahead\n')
rows = {b['name']: b for b in r._branches()}
test_eq(rows['main']['upstream'], 'origin/main')
test_eq((rows['main']['ahead'], rows['main']['behind']), (1, 0))
assert 'origin/main' not in rows                        # claimed by a local branch, so not shown twice

sh(r.root, 'push', 'origin', 'main:unclaimed')
sh(r.root, 'fetch', 'origin')
assert 'origin/unclaimed' in {b['name'] for b in r._branches()}

## Filters that are configured and missing

A repository can name a clean filter in `.gitattributes` that is not installed in the environment
Git runs in. Git does not treat a failing filter as an error, so the affected files reappear as
modified however often they are staged. `filter_health` is what turns that into a sentence somebody
can act on.

In [ ]:
#| export
@patch
def filter_health(self:GitRepo):
    "Whether this repository's content filters can actually run."
    def probe():
        names = set()
        for p in (self.root/'.gitattributes', self._gitdir()/'info'/'attributes'):
            try:
                names.update(re.findall(r'filter=([\w.+-]+)', p.read_text(encoding='utf-8',
                    errors='replace')))
            except OSError:
                continue
        rows = []
        for name in sorted(names):
            def config(field): return self._ask('config', '--get', f'filter.{name}.{field}')
            clean, smudge = config('clean'), config('smudge')
            command = clean or smudge
            resolved = gateway().resolves(self.root, command) if command else ''
            rows.append({
                'name': name, 'clean': clean, 'smudge': smudge,
                'required': config('required').lower() == 'true', 'resolved': resolved,
                'configured': bool(command),
                'ok': bool(command) and bool(resolved),
            })
        broken = [r for r in rows if not r['ok']]
        return {'filters': rows, 'ok': not broken,
            'explain': _explain_filters(broken) if broken else ''}
    return gateway().memo(self.root, 'filters', probe)

In [ ]:
r = GitRepo.at(mkrepo())
test_eq(r._gitdir(), r.root/'.git')
health = r.filter_health()
test_eq((health['filters'], health['ok'], health['explain']), ([], True, ''))

# The answer is memoised per repository, so each case below gets one of its own.
c = GitRepo.at(mkrepo({'.gitattributes': '*.ipynb filter=nbstripout\n'}))
sh(c.root, 'config', 'filter.nbstripout.clean', 'definitely-not-installed')
health = c.filter_health()
test_eq([b['name'] for b in health['filters']], ['nbstripout'])
test_eq((health['ok'], health['filters'][0]['configured']), (False, True))
test_eq(health['filters'][0]['resolved'], '')
assert 'cannot be found' in health['explain']

u = GitRepo.at(mkrepo({'.gitattributes': '*.bin filter=missing-test-filter\n'}))
health = u.filter_health()
test_eq([b['name'] for b in health['filters']], ['missing-test-filter'])
test_eq((health['ok'], health['filters'][0]['configured']), (False, False))
assert 'configured nowhere' in health['explain']

## Staging

The four one-liners are `git add` and `git restore` under the names an interface uses.
`write_worktree` and `apply_patch` are the two ways an edit reaches the tree, and `ignore` appends
to `.gitignore` without writing the same line twice.

In [ ]:
#| export
@patch
def stage(self:GitRepo, paths): self._mutate('add', '--', *paths)

@patch
def unstage(self:GitRepo, paths): self._mutate('restore', '--staged', '--', *paths)

@patch
def discard(self:GitRepo, paths): self._mutate('restore', '--worktree', '--', *paths)

@patch
def ignore(self:GitRepo, path, directory=False):
    "Add one repository-relative path to .gitignore without duplicating an existing rule."
    rule = str(path).strip().strip('/') + ('/' if directory else '')
    if not rule or rule.startswith('../') or '/..' in rule: raise GitError('invalid ignore path')
    target = self.root/'.gitignore'
    lines = target.read_text(encoding='utf-8').splitlines() if target.exists() else []
    if rule not in lines:
        target.write_text('\n'.join(lines + [rule]) + '\n', encoding='utf-8')
        invalidate(self.root)
    return {'path': str(target), 'rule': rule}

@patch
def write_worktree(self:GitRepo, path, content):
    target = (self.root/str(path)).resolve()
    if not target.is_relative_to(self.root.resolve()):
        raise GitError('file is outside the repository')
    if target.exists() and not target.is_file(): raise GitError(f'{path} is not a working-tree file')
    if not target.parent.exists():
        raise GitError(f'the parent directory for {path} does not exist')
    target.write_text(str(content), encoding='utf-8')
    invalidate(self.root)

@patch
def apply_patch(self:GitRepo, patch, staged=True, reverse=False):
    if not str(patch).strip(): raise GitError('select at least one diff hunk')
    args = ['apply', '--whitespace=nowarn']
    if staged: args.append('--cached')
    if reverse: args.append('--reverse')
    self._mutate(*args, input=str(patch))

In [ ]:
r = GitRepo.at(mkrepo())
write(r.root, 'a.txt', 'a\n')
r.stage(['a.txt'])
test_eq([c['staged'] for c in r.changes()], [True])
r.unstage(['a.txt'])
test_eq([c['untracked'] for c in r.changes()], [True])

write(r.root, 'readme.md', 'changed\n')
r.discard(['readme.md'])
test_eq((r.root/'readme.md').read_text(), 'hello\n')

test_eq(r.ignore('a.txt')['rule'], 'a.txt')
test_eq((r.root/'.gitignore').read_text(), 'a.txt\n')
r.ignore('a.txt')
test_eq((r.root/'.gitignore').read_text(), 'a.txt\n')   # not written twice
test_eq(r.ignore('build', directory=True)['rule'], 'build/')
test_fail(lambda: r.ignore('../escape'), contains='invalid ignore path')

r.write_worktree('readme.md', 'written\n')
test_eq((r.root/'readme.md').read_text(), 'written\n')
test_fail(lambda: r.write_worktree('../escape.txt', 'x'), contains='outside')
test_fail(lambda: r.write_worktree('nowhere/b.txt', 'x'), contains='does not exist')

r.discard(['readme.md'])
r.apply_patch(_unified('hello\n', 'patched\n', 'a/readme.md', 'b/readme.md'))
test_eq(r._ask('show', ':readme.md'), 'patched')        # staged by default: the index moves
test_eq((r.root/'readme.md').read_text(), 'hello\n')    # ...and the worktree does not
test_fail(lambda: r.apply_patch('  '), contains='at least one diff hunk')

## Switching branches

`checkout` accepts a local branch or a remote-tracking branch. A remote branch creates or updates a local tracking branch.

If Git refuses the switch because it would overwrite local changes, `checkout` sets those changes aside and retries. If the changes conflict with the target branch, it restores the original branch and working tree.

In [ ]:
#| export
@patch
def _has_ref(self:GitRepo, ref):
    return not _run(self.root, 'show-ref', '--verify', '--quiet', ref, check=False).returncode

@patch
def _switch(self:GitRepo, branch):
    local = branch.partition('/')[2]
    if not self._has_ref(f'refs/remotes/{branch}'):
        self._mutate('switch', branch)
    elif self._has_ref(f'refs/heads/{local}'):
        self._mutate('switch', local)
        self._mutate('branch', f'--set-upstream-to={branch}', local)
    else:
        self._mutate('switch', '--track', branch)
    return ''

@patch
def checkout(self:GitRepo, branch):
    "Switch branches, setting aside uncommitted work only if Git refuses to carry it."
    if not str(branch).strip(): raise GitError('choose a branch')
    def call():
        try:
            return self._switch(branch)
        except GitError as e:
            if not any(w in str(e).lower() for w in self._WOULD_CLOBBER):
                raise
            here = self.run('branch', '--show-current').strip()
            aside = self._set_aside('checkout')
            if not aside:
                raise
            self._switch(branch)
            if not (note := self._bring_back(aside)):
                return f'your uncommitted work was carried to {branch}.'
            self._switch(here)
            self._bring_back(aside)
            raise GitError(
                f'cannot switch to {branch} without losing your uncommitted changes -- they '
                f'conflict with what is on that branch. Nothing was changed; you are still '
                f'on {here} with your work. Commit or stash it first.') from e
    return self._guarded('checkout', call)

In [ ]:
r = GitRepo.at(mkrepo())
assert r._has_ref('refs/heads/main')
assert not r._has_ref('refs/heads/nope')

sh(r.root, 'branch', 'feature')
out = r.checkout('feature')
test_eq(r.run('branch', '--show-current').strip(), 'feature')
test_eq(out['op'], 'checkout')
test_fail(lambda: r.checkout('nope'), contains='nope')

bare = mkbare()
sh(r.root, 'remote', 'add', 'origin', str(bare))
sh(r.root, 'push', 'origin', 'main:published')
sh(r.root, 'fetch', 'origin')
r.checkout('origin/published')
test_eq(r.run('branch', '--show-current').strip(), 'published')
test_eq(r._tracking()[0], 'origin/published')

# Uncommitted work the switch cannot carry leaves the repository exactly where it was.
r.checkout('main')
commit(r.root, 'diverge', readme__md='on main\n')
r.checkout('feature')
write(r.root, 'readme.md', 'in progress\n')
test_fail(lambda: r.checkout('main'), contains='you are still on feature with your work')
test_eq(r.run('branch', '--show-current').strip(), 'feature')
test_eq((r.root/'readme.md').read_text(), 'in progress\n')

# ...and work that has nowhere to clash comes across with it.
r.discard(['readme.md'])
write(r.root, 'scratch.txt', 'notes\n')
r.stage(['scratch.txt'])
r.checkout('main')
test_eq(r.run('branch', '--show-current').strip(), 'main')
test_eq((r.root/'scratch.txt').read_text(), 'notes\n')

## Branch porcelain

The rest of `git branch`, one line each. `delete_remote` is a push, so it takes the longer timeout a
network command needs.

In [ ]:
#| export
@patch
def create(self:GitRepo, branch, start='HEAD'): self._mutate('switch', '-c', branch, start)

@patch
def rename_branch(self:GitRepo, old, new): self._mutate('branch', '-m', old, new)

@patch
def set_upstream(self:GitRepo, branch, upstream): self._mutate('branch', f'--set-upstream-to={upstream}', branch)

@patch
def unset_upstream(self:GitRepo, branch): self._mutate('branch', '--unset-upstream', branch)

@patch
def delete(self:GitRepo, branch, force=False): self._mutate('branch', '-D' if force else '-d', branch)

@patch
def delete_remote(self:GitRepo, remote, branch): self._mutate('push', remote, '--delete', branch, timeout=120)

In [ ]:
r = GitRepo.at(mkrepo())
r.create('feature')
test_eq(r.run('branch', '--show-current').strip(), 'feature')
r.create('from-main', 'main')
test_eq(r._ask('rev-parse', 'from-main'), r._ask('rev-parse', 'main'))

r.checkout('main')
r.rename_branch('from-main', 'renamed')
assert 'renamed' in {b['name'] for b in r._branches()}

bare = mkbare()
sh(r.root, 'remote', 'add', 'origin', str(bare))
sh(r.root, 'push', 'origin', 'main')
sh(r.root, 'fetch', 'origin')
r.set_upstream('renamed', 'origin/main')
test_eq({b['name']: b['upstream'] for b in r._branches()}['renamed'], 'origin/main')
r.unset_upstream('renamed')
test_eq({b['name']: b['upstream'] for b in r._branches()}['renamed'], '')

r.delete('renamed')
assert 'renamed' not in {b['name'] for b in r._branches()}
sh(r.root, 'branch', 'gone-soon')
r.delete('gone-soon', force=True)
assert 'gone-soon' not in {b['name'] for b in r._branches()}

sh(r.root, 'push', 'origin', 'main:doomed')
sh(r.root, 'fetch', 'origin')
r.delete_remote('origin', 'doomed')
assert 'doomed' not in sh(bare, 'branch', '--list')

## Commit, and the remote

Four mutations that all return the same outcome shape. `push` publishes an unpublished branch when
asked, and `pull` is fast-forward only: a merge a person did not ask for is not a pull.

In [ ]:
#| export
@patch
def commit(self:GitRepo, message, amend=False):
    if not message.strip(): raise GitError('a commit message is required')
    args = ['commit', *(['--amend'] if amend else []), '-m', message.strip()]
    return self._guarded('commit', lambda: self._mutate(*args).strip())

@patch
def fetch(self:GitRepo, remote='', prune=True):
    args = ['fetch', remote] if remote else ['fetch', '--all']
    if prune: args.append('--prune')
    return self._mutate(*args, timeout=120).strip()

@patch
def pull(self:GitRepo):
    return self._guarded('pull', lambda: self._mutate(
        'pull', '--ff-only', '--autostash', timeout=120).strip())

@patch
def push(self:GitRepo, publish=False, force_with_lease=False):
    args = ['push']
    if publish:
        branch = self.run('branch', '--show-current').strip()
        if not branch: raise GitError('cannot publish a detached HEAD')
        remotes = self._remote_names()
        if not remotes: raise GitError('add a remote before publishing this branch')
        args += ['--set-upstream', 'origin' if 'origin' in remotes else remotes[0], branch]
    if force_with_lease: args.append('--force-with-lease')
    return self._mutate(*args, timeout=120).strip()

In [ ]:
r = GitRepo.at(mkrepo())
write(r.root, 'a.txt', 'a\n')
r.stage(['a.txt'])
out = r.commit('add a')
test_eq(out['op'], 'commit')
test_eq(out['moved'], True)
test_eq(r._ask('log', '-1', '--format=%s'), 'add a')

r.commit('add a, said better', amend=True)
test_eq(r._ask('log', '-1', '--format=%s'), 'add a, said better')

bare = mkbare()
sh(r.root, 'remote', 'add', 'origin', str(bare))
r.push(publish=True)                                    # `push` and `fetch` report as text
test_eq(r._tracking()[0], 'origin/main')
r.fetch()
test_eq(r.pull()['op'], 'pull')                         # ...and `pull` as an outcome, being guarded
test_fail(lambda: GitRepo.at(mkrepo()).push(publish=True), contains='add a remote')

# A second clone commits, and the first pulls it down as a fast-forward.
other = Path(tempfile.mkdtemp()); _tmp.append(str(other))
o = GitRepo.at(clone(str(bare), other, 'other'))
sh(o.root, 'config', 'user.email', 'tests@example.com')
sh(o.root, 'config', 'user.name', 'Repo tests')
commit(o.root, 'from the other clone', b__txt='b\n')
o.push()
test_eq(r.pull()['moved'], True)
test_eq((r.root/'b.txt').read_text(), 'b\n')

## Stash

`stash` includes untracked files, because work in progress usually is some. The rest name the ref
they act on, so an interface can offer a list rather than only the top of the pile.

In [ ]:
#| export
@patch
def stash(self:GitRepo, message=''): return self._mutate('stash', 'push', '-u', *(['-m', message] if message else [])).strip()

@patch
def stash_pop(self:GitRepo, ref='stash@{0}'):
    return self._guarded('stash pop', lambda: self._attempt('stash', 'pop', ref))

@patch
def stash_apply(self:GitRepo, ref='stash@{0}'):
    return self._guarded('stash apply', lambda: self._attempt('stash', 'apply', ref))

@patch
def stash_drop(self:GitRepo, ref='stash@{0}'): return self._mutate('stash', 'drop', ref).strip()

In [ ]:
r = GitRepo.at(mkrepo())
write(r.root, 'readme.md', 'in progress\n')
write(r.root, 'untracked.txt', 'also\n')
r.stash('wip')
test_eq((r.root/'readme.md').read_text(), 'hello\n')
assert not (r.root/'untracked.txt').exists()            # -u: untracked work is work
assert 'wip' in sh(r.root, 'stash', 'list')

r.stash_pop()
test_eq((r.root/'readme.md').read_text(), 'in progress\n')
test_eq(sh(r.root, 'stash', 'list'), '')

r.stash('again')
r.stash_apply()
test_eq((r.root/'readme.md').read_text(), 'in progress\n')
assert 'again' in sh(r.root, 'stash', 'list')           # applied, not popped
r.stash_drop()
test_eq(sh(r.root, 'stash', 'list'), '')

## Bringing work together

Merge, rebase, cherry-pick, revert and reset, each guarded and each reporting a conflict as an
outcome rather than raising. `operation_action` is what continues, skips or aborts whichever of them
stopped.

In [ ]:
#| export
@patch
def merge(self:GitRepo, branch, strategy='merge'):
    "Merge `branch`, setting uncommitted work aside for the length of it."
    if strategy not in {'merge', 'squash', 'ff-only'}:
        raise GitError('merge strategy must be merge, squash, or ff-only')
    args = ['merge', '--no-edit']
    if strategy == 'squash': args.append('--squash')
    elif strategy == 'ff-only': args.append('--ff-only')
    return self._guarded('merge', lambda: self._attempt(*args, branch), autostash=True)

@patch
def rebase(self:GitRepo, branch):
    return self._guarded('rebase', lambda: self._attempt('rebase', '--autostash', branch))

@patch
def cherry_pick(self:GitRepo, ref):
    return self._guarded('cherry-pick', lambda: self._attempt('cherry-pick', ref))

@patch
def revert(self:GitRepo, ref):
    return self._guarded('revert', lambda: self._attempt('revert', '--no-edit', ref))

@patch
def reset(self:GitRepo, ref='HEAD', mode='mixed'):
    if mode not in {'soft', 'mixed', 'hard'}:
        raise GitError('reset mode must be soft, mixed, or hard')
    return self._guarded('reset', lambda: self._mutate('reset', f'--{mode}', ref).strip())

@patch
def operation_action(self:GitRepo, action):
    active = self._operation()['active']
    if not active: raise GitError('no Git operation is in progress')
    if action not in {'continue', 'skip', 'abort'}:
        raise GitError('operation action must be continue, skip, or abort')
    command = active if active in {'cherry-pick', 'revert', 'rebase', 'merge'} else ''
    if not command or (action == 'skip' and active not in {'rebase', 'cherry-pick'}):
        raise GitError(f'cannot {action} the active {active}')
    return self._guarded(f'{active} --{action}',
        lambda: self._mutate(command, f'--{action}').strip())

In [ ]:
r = GitRepo.at(mkrepo())
sh(r.root, 'branch', 'feature')
commit(r.root, 'on main', main__txt='main\n')
r.checkout('feature')
commit(r.root, 'on feature', feature__txt='feature\n')

out = r.merge('main')
test_eq(out['op'], 'merge')
test_eq(out['conflicted'], [])
assert (r.root/'main.txt').exists()

# A conflict is an outcome: the files are named and the operation is left in progress.
c = GitRepo.at(mkrepo({'shared.txt': 'base\n'}))
sh(c.root, 'branch', 'other')
commit(c.root, 'ours', shared__txt='ours\n')
c.checkout('other')
commit(c.root, 'theirs', shared__txt='theirs\n')
out = c.merge('main')
test_eq(out['conflicted'], ['shared.txt'])
test_eq(out['operation']['active'], 'merge')
assert 'resolve them to continue' in out['summary']
aborted = c.operation_action('abort')
test_eq((aborted['op'], aborted['operation']['active']), ('merge --abort', ''))

test_fail(lambda: c.operation_action('abort'), contains='no Git operation is in progress')
test_fail(lambda: c.merge('main', 'sideways'), contains='merge strategy must be')

r2 = GitRepo.at(mkrepo())
sh(r2.root, 'branch', 'topic')
commit(r2.root, 'on main', m__txt='m\n')
r2.checkout('topic')
commit(r2.root, 'on topic', t__txt='t\n')
test_eq(r2.rebase('main')['op'], 'rebase')
test_eq(r2._ask('log', '--format=%s', '-3').splitlines(), ['on topic', 'on main', 'initial'])

pick = r2._ask('rev-parse', 'main')
r2.checkout('main')
test_eq(r2.revert(r2._ask('rev-parse', 'HEAD'))['op'], 'revert')
assert not (r2.root/'m.txt').exists()

head = r2._ask('rev-parse', 'HEAD')
test_eq(r2.reset('HEAD~1', 'hard')['op'], 'reset')
test_ne(r2._ask('rev-parse', 'HEAD'), head)
test_fail(lambda: r2.reset('HEAD', 'sideways'), contains='mode')

r3 = GitRepo.at(mkrepo())
sh(r3.root, 'branch', 'side')
commit(r3.root, 'on main', only__txt='only\n')
target = r3._ask('rev-parse', 'HEAD')
r3.checkout('side')
test_eq(r3.cherry_pick(target)['op'], 'cherry-pick')
assert (r3.root/'only.txt').exists()

## Tags and remotes

The rest of the porcelain: annotated and lightweight tags, deleting one locally or on a remote, and
the three remote commands.

In [ ]:
#| export
@patch
def tag(self:GitRepo, name, message='', ref='HEAD'):
    if not name.strip(): raise GitError('a tag name is required')
    args = ['tag']
    if message.strip():
        args += ['-a', name.strip(), '-m', message.strip(), ref]
    else:
        args += [name.strip(), ref]
    return self._mutate(*args).strip()

@patch
def delete_tag(self:GitRepo, name, remote=''):
    self._mutate('tag', '-d', name)
    if remote: self._mutate('push', remote, f':refs/tags/{name}', timeout=120)

@patch
def add_remote(self:GitRepo, name, url): self._mutate('remote', 'add', name, url)

@patch
def set_remote(self:GitRepo, name, url, push=False): self._mutate('remote', 'set-url', *(['--push'] if push else []), name, url)

@patch
def remove_remote(self:GitRepo, name): self._mutate('remote', 'remove', name)

In [ ]:
r = GitRepo.at(mkrepo())
r.tag('v1')
test_eq(sh(r.root, 'tag', '--list'), 'v1')
test_eq(r._ask('cat-file', '-t', 'v1'), 'commit')       # lightweight
r.tag('v2', 'the second one')
test_eq(r._ask('cat-file', '-t', 'v2'), 'tag')          # annotated
r.delete_tag('v2')
test_eq(sh(r.root, 'tag', '--list'), 'v1')

bare = mkbare()
r.add_remote('origin', str(bare))
test_eq(r._remote_names(), ['origin'])
r.set_remote('origin', 'https://example.invalid/o/r.git')
test_eq(r._remotes()[0]['fetch'], 'https://example.invalid/o/r.git')
r.set_remote('origin', str(bare))
sh(r.root, 'push', 'origin', 'v1')
assert 'v1' in sh(bare, 'tag', '--list')
r.delete_tag('v1', remote='origin')
test_eq(sh(bare, 'tag', '--list'), '')
r.remove_remote('origin')
test_eq(r._remote_names(), [])

## Repository status for an interface

`info` returns the current branch, upstream divergence, branches, remotes, changes, stashes, tags, active operation, filter health, and recent safepoints. The result is cached briefly so one panel refresh can reuse it.

In [ ]:
#| export
@patch
def info(self:GitRepo):
    def collect():
        branch = self.run('branch', '--show-current').strip()
        oid = self._ask('rev-parse', '--short', 'HEAD')
        if not branch:
            branch = (oid + ' (detached)') if oid else 'HEAD (unborn)'
        upstream_name, ahead, behind = self._tracking()
        changes = self._changes_uncached()
        stashes = [{'ref': ref, 'subject': subject, 'timestamp': int(stamp or 0)}
            for ref, subject, stamp in
            (_fields(l, 3) for l in self.run('stash', 'list', '--format=%gd%x00%gs%x00%ct').splitlines())]
        tag_fmt = '%(refname:short)%00%(objectname:short)%00%(creatordate:unix)%00%(subject)'
        tag_details = [{'name': name, 'oid': tag_oid, 'timestamp': int(stamp or 0), 'subject': subject}
            for name, tag_oid, stamp, subject in (_fields(l, 4) for l in
            self.run('tag', '-l', f'--format={tag_fmt}', '--sort=-creatordate').splitlines()[:100])]
        return {
            'root': str(self.root), 'branch': branch, 'head': oid, 'upstream': upstream_name,
            'ahead': ahead, 'behind': behind, 'clean': not changes, 'changes': changes,
            'branches': self._branches(), 'remotes': self._remotes(), 'stashes': stashes,
            'tags': [tag['name'] for tag in tag_details], 'tag_details': tag_details,
            'operation': self._operation(), 'fetched_at': self._fetch_time(),
            'filters': self.filter_health(),
            'safepoints': self.safepoints()[:12],
        }
    value = _cached(self.root, 'info', collect)
    with _CACHE_LOCK:
        _CACHE[(str(self.root), 'changes')] = (time.monotonic(), value['changes'])
    return value

@patch
def _fetch_time(self:GitRepo):
    fetch_head = Path(self.run('rev-parse', '--git-path', 'FETCH_HEAD').strip())
    if not fetch_head.is_absolute():
        fetch_head = self.root/fetch_head
    try:
        return int(fetch_head.stat().st_mtime)
    except OSError:
        return 0

In [ ]:
r = GitRepo.at(mkrepo())
d = r.info()
test_eq(d['root'], str(r.root))
test_eq(d['branch'], 'main')
test_eq((d['clean'], d['upstream'], d['ahead'], d['behind']), (True, '', 0, 0))
test_eq([b['name'] for b in d['branches']], ['main'])
test_eq((d['changes'], d['remotes']), ([], []))
test_eq(d['operation']['active'], '')
test_eq(r._fetch_time(), 0)                             # nothing has been fetched yet

write(r.root, 'readme.md', 'changed\n')
test_eq(r.info()['clean'], True)                        # `info` is cached for less than a second
invalidate(r.root)
test_eq(r.info()['clean'], False)
test_eq([c['path'] for c in r.info()['changes']], ['readme.md'])

bare = mkbare()
sh(r.root, 'remote', 'add', 'origin', str(bare))
sh(r.root, 'push', '-u', 'origin', 'main')
sh(r.root, 'fetch', 'origin')
invalidate(r.root)
test_eq(r.info()['upstream'], 'origin/main')
assert r._fetch_time() > 0                              # ...and now something has

## Compact repository status

`brief` returns a smaller status record for prompts and summaries. It includes branch divergence, change counts, the last commit, the latest tag, the active operation, and the origin URL. Pass `fresh=True` to bypass the short-lived cache.

In [ ]:
#| export
@patch
def brief(self:GitRepo, fresh=False):
    "One repository as a single row: branch, drift, dirt, and how far past its last tag."
    if fresh: invalidate(self.root)
    def collect():
        branch = self.run('branch', '--show-current').strip()
        detached = not branch
        if detached:
            branch = self._ask('rev-parse', '--short', 'HEAD') or 'HEAD (unborn)'
        upstream_name, ahead, behind = self._tracking()
        changes = self._changes_uncached()
        short, subject, stamp = _fields(self._ask('log', '-1', '--format=%h%x00%s%x00%ct'), 3)
        tag = self._ask('describe', '--tags', '--abbrev=0')
        counted = self._ask('rev-list', '--count', f'{tag}..HEAD') if tag else ''
        unreleased = int(counted) if counted else None
        origin = self._ask('remote', 'get-url', 'origin')
        return {
            'root': str(self.root), 'name': self.root.name, 'branch': branch,
            'detached': detached, 'upstream': upstream_name, 'ahead': ahead, 'behind': behind,
            'changed': len(changes), 'clean': not changes,
            'staged': sum(1 for c in changes if c['staged']),
            'unstaged': sum(1 for c in changes if c['unstaged'] and not c['untracked']),
            'untracked': sum(1 for c in changes if c['untracked']),
            'conflicted': sum(1 for c in changes if c['conflicted']),
            'last_commit': {'short': short, 'subject': subject, 'timestamp': int(stamp or 0)},
            'tag': tag, 'unreleased': unreleased, 'operation': self._operation()['active'],
            'fetched_at': self._fetch_time(), 'remote': origin,
            'web_url': _remote_web_url(origin),
        }
    return _cached(self.root, 'brief', collect)

In [ ]:
r = GitRepo.at(mkrepo())
b = r.brief()
test_eq((b['branch'], b['clean'], b['changed']), ('main', True, 0))
test_eq(b['name'], r.root.name)
test_eq(b['last_commit']['subject'], 'initial')
test_eq((b['upstream'], b['ahead'], b['behind'], b['operation']), ('', 0, 0, ''))
test_eq(r.brief(), r.brief())

write(r.root, 'readme.md', 'changed\n')
test_eq(r.brief()['clean'], True)                       # the cached answer, from before the edit
b = r.brief(fresh=True)
test_eq((b['clean'], b['changed'], b['unstaged']), (False, 1, 1))

## History

One `log` with a NUL-separated format, one `reflog`, and one commit read in full. `_resolve_ref`
turns whatever a caller wrote into an oid, and answers with an empty string rather than raising
when there is nothing there.

In [ ]:
#| export
@patch
def history(self:GitRepo, limit=100, skip=0, query='', ref='--all', path=''):
    limit, skip = max(1, min(int(limit), 500)), max(0, int(skip))
    fmt = '%H%x00%h%x00%P%x00%an%x00%ae%x00%at%x00%D%x00%G?%x00%s'
    args = ['log', ref, f'--max-count={limit}', f'--skip={skip}', f'--format={fmt}']
    if query: args += ['--regexp-ignore-case', f'--grep={query}']
    if path: args += ['--', path]
    rows = []
    for line in self.run(*args).splitlines():
        (oid, short, parents, author, email,
            stamp, decoration, signature, subject) = _fields(line, 9)
        rows.append({'oid': oid, 'short': short, 'parents': parents.split(), 'author': author,
            'email': email, 'timestamp': int(stamp or 0), 'decoration': decoration,
            'signature': signature, 'subject': subject})
    return rows

@patch
def reflog(self:GitRepo, limit=100):
    fmt = '%H%x00%h%x00%gD%x00%gs%x00%an%x00%at'
    rows = []
    for line in self.run('reflog', f'--max-count={max(1, min(int(limit), 500))}', f'--format={fmt}').splitlines():
        oid, short, selector, subject, author, stamp = _fields(line, 6)
        rows.append({'oid': oid, 'short': short, 'selector': selector, 'subject': subject,
            'author': author, 'timestamp': int(stamp or 0)})
    return rows

@patch
def commit_detail(self:GitRepo, ref):
    meta = self.run('show', '-s', '--format=%H%x00%h%x00%P%x00%an%x00%ae%x00%at%x00%D%x00%B', ref)
    (oid, short, parents, author, email,
        stamp, decoration, message) = _fields(meta.rstrip('\n'), 8)
    patch = self.run('show', '--format=', '--no-ext-diff', '--stat', '--patch', ref)
    return {'oid': oid, 'short': short, 'parents': parents.split(), 'author': author,
        'email': email, 'timestamp': int(stamp or 0), 'decoration': decoration,
        'message': message.strip(), 'patch': patch}

@patch
def _resolve_ref(self:GitRepo, ref):
    "Resolve a user-facing ref to a commit without allowing option-like refs."
    ref = str(ref or '').strip()
    if not ref: raise GitError('choose a branch, tag, or commit')
    found = _run(self.root, 'rev-parse', '--verify', '--end-of-options',
        f'{ref}^{{commit}}', check=False)
    if found.returncode: raise GitError(f'unknown branch, tag, or commit: {ref}')
    return found.stdout.strip()

In [ ]:
r = GitRepo.at(mkrepo())
second = commit(r.root, 'the second commit', a__txt='a\n')
rows = r.history()
test_eq([x['subject'] for x in rows], ['the second commit', 'initial'])
test_eq(rows[0]['short'], second)
test_eq(rows[0]['author'], 'Repo tests')
test_eq(len(rows[0]['oid']), 40)

test_eq(len(r.history(limit=1)), 1)
test_eq([x['subject'] for x in r.history(skip=1)], ['initial'])
test_eq([x['subject'] for x in r.history(query='second')], ['the second commit'])
test_eq([x['subject'] for x in r.history(path='a.txt')], ['the second commit'])
test_eq(r.history(ref='HEAD~1..HEAD')[0]['subject'], 'the second commit')

assert r.reflog()
test_eq(r.reflog(limit=1) and len(r.reflog(limit=1)), 1)

d = r.commit_detail('HEAD')
test_eq(d['message'], 'the second commit')
test_eq(d['short'], second)
test_eq(d['parents'], [r.history()[-1]['oid']])
assert 'a.txt' in d['patch']

test_eq(len(r._resolve_ref('HEAD')), 40)
test_eq(r._resolve_ref('HEAD~1'), r.history()[-1]['oid'])
test_fail(lambda: r._resolve_ref('nope'), contains='unknown branch, tag, or commit')
test_fail(lambda: r._resolve_ref(''), contains='choose a branch')
test_fail(lambda: r._resolve_ref('--upload-pack=evil'), contains='unknown branch')

## Comparing refs safely

`_diff_files` returns rename-aware file records for two refs. `_merge_tree` uses `git merge-tree --write-tree` to rehearse a merge. It can write loose objects, but it does not change refs, the index, or the working tree.

In [ ]:
#| export
@patch
def _diff_files(self:GitRepo, left, right):
    "Per-file records for one comparison: rename-aware, with line counts merged in."
    rows = _diff_status(self.run('diff', '--name-status', '-z', '-M', left, right, '--'))
    stats = _diff_numstat(self.run('diff', '--numstat', '-z', '-M', left, right, '--'))
    for row in rows:
        row.update(stats.get(row['path'], {'additions': 0, 'deletions': 0, 'binary': False}))
    return rows

@patch
def _merge_tree(self:GitRepo, ours, theirs, base=''):
    "One in-memory merge: the tree it wrote, the files that would conflict, and whether any do."
    args = ['merge-tree', '--write-tree', '--name-only', *([f'--merge-base={base}'] if base else [])]
    r = _run(self.root, *args, ours, theirs, check=False)
    lines = r.stdout.splitlines()
    tree = lines[0].strip() if lines else ''
    if not r.returncode: return tree, [], False
    named = []
    for x in lines[1:]:                     # the file list ends at the blank line before the messages
        if not x.strip(): break
        named.append(x.strip())
    return tree, uniqueify(named), True

@patch
def _rehearse(self:GitRepo, ours, theirs, base=''):
    "Merge `theirs` into `ours` in memory: the files that would conflict, and whether any do."
    _, files, clashed = self._merge_tree(ours, theirs, base)
    return files, clashed

@patch
def _side_by_side(self:GitRepo, path, left, right, from_label, to_label, patch):
    "Both versions of one file and the patch between them, decoded when it is a notebook."
    if not str(path).lower().endswith('.ipynb'): return left, right, patch
    left, right = _notebook_content(left), _notebook_content(right)
    return left, right, _unified(left, right, f'{path} ({from_label})', f'{path} ({to_label})')

In [ ]:
r = GitRepo.at(mkrepo({'a.txt': 'one\n'}))
base = r._ask('rev-parse', 'HEAD')
commit(r.root, 'change a, add b', a__txt='two\n', b__txt='b\n')
rows = {x['path']: x for x in r._diff_files(base, 'HEAD')}
test_eq(sorted(rows), ['a.txt', 'b.txt'])
test_eq((rows['a.txt']['additions'], rows['a.txt']['deletions']), (1, 1))
test_eq(rows['b.txt']['status'], 'A')

m = GitRepo.at(mkrepo({'shared.txt': 'base\n'}))
sh(m.root, 'branch', 'other')
commit(m.root, 'ours', shared__txt='ours\n')
ours = m._ask('rev-parse', 'HEAD')
m.checkout('other')
commit(m.root, 'theirs', shared__txt='theirs\n')
theirs = m._ask('rev-parse', 'HEAD')

tree, files, clashed = m._merge_tree(ours, theirs)
test_eq((files, clashed), (['shared.txt'], True))
assert tree
test_eq(m._rehearse(ours, theirs), (['shared.txt'], True))
test_eq(m._ask('rev-parse', 'HEAD'), theirs)            # rehearsing moved nothing

clean = GitRepo.at(mkrepo({'a.txt': 'a\n'}))
sh(clean.root, 'branch', 'other')
commit(clean.root, 'ours', a__txt='ours\n')
o = clean._ask('rev-parse', 'HEAD')
clean.checkout('other')
commit(clean.root, 'theirs', b__txt='b\n')
test_eq(clean._rehearse(o, 'HEAD'), ([], False))

## Reviewing a comparison

`review` is a whole comparison as a person reads it: the files, their counts, and a side-by-side pair
per file.

In [ ]:
#| export
@patch
def review(self:GitRepo, base, head, mode='review'):
    "Per-file review metadata: `review` compares the merge base, `snapshot` the whole trees."
    if mode not in {'review', 'snapshot'}:
        raise GitError('comparison mode must be review or snapshot')
    base_name, head_name = str(base).strip(), str(head).strip()
    base_oid, head_oid = self._resolve_ref(base_name), self._resolve_ref(head_name)
    merge_base = self._ask('merge-base', base_oid, head_oid)
    if not merge_base:
        raise GitError(f'{base_name} and {head_name} do not share a merge base')
    diff_left = merge_base if mode == 'review' else base_oid
    status = self._diff_files(diff_left, head_oid)
    counts = self.run('rev-list', '--left-right', '--count', f'{base_oid}...{head_oid}').split()
    alternate = None
    if mode == 'review' and not status and base_oid != head_oid:
        direct = _summarise_diff(self._diff_files(base_oid, head_oid))
        alternate = {k: direct[k] for k in ('files', 'additions', 'deletions')}
    head_now = self._ask('rev-parse', '--verify', 'HEAD')
    return {'base': base_name, 'head': head_name, 'base_oid': base_oid,
        'head_oid': head_oid, 'diff_base_oid': diff_left, 'merge_base': merge_base,
        'mode': mode, 'base_only': int(counts[0]), 'head_only': int(counts[1]),
        'files': status, 'snapshot_alternative': alternate,
        'commits': self.history(limit=250, ref=f'{base_oid}..{head_oid}'),
        'summary': _summarise_diff(status),
        'can_apply': base_oid == head_now and merge_base == base_oid and not self.changes()}

In [ ]:
r = GitRepo.at(mkrepo({'a.txt': 'one\n'}))
base = r._ask('rev-parse', 'HEAD')
commit(r.root, 'change a, add b', a__txt='two\n', b__txt='b\n')

rev = r.review(base, 'HEAD')
test_eq(sorted(f['path'] for f in rev['files']), ['a.txt', 'b.txt'])
test_eq(rev['summary'], {'files': 2, 'additions': 2, 'deletions': 1, 'binary': 0})
test_eq((rev['base_only'], rev['head_only']), (0, 1))
test_eq(rev['merge_base'], base)
test_eq([c['subject'] for c in rev['commits']], ['change a, add b'])
test_fail(lambda: r.review(base, 'HEAD', 'sideways'), contains='review or snapshot')

## One file of it

`review_file` is one file of a comparison, for a pane that loads on demand rather than a review that
loads every file up front.

In [ ]:
#| export
@patch
def _ref_file(self:GitRepo, ref, path):
    if not ref or not path: return ''
    result = _run(self.root, 'show', f'{ref}:{path}', check=False)
    return result.stdout if result.returncode == 0 else ''

@patch
def _file_pair(self:GitRepo, path, base, head, from_label, to_label, context, old_path=''):
    "Both sides of one file between two commits, and the patch between them."
    patch = self.run('diff', '--no-ext-diff', _unified_arg(context), '-M', base, head, '--', path)
    left = self._ref_file(base, old_path or path)
    right = self._ref_file(head, path)
    return self._side_by_side(path, left, right, from_label, to_label, patch)

@patch
def review_file(self:GitRepo, base, head, path, mode='review', context=3):
    review = self.review(base, head, mode)
    known = first(x for x in review['files'] if x['path'] == path)
    if known is None: raise GitError(f'{path} is not changed in this comparison')
    left, right, patch = self._file_pair(path, review['diff_base_oid'], review['head_oid'],
        review['base'], review['head'], context, known.get('old_path'))
    return {'file': known, 'patch': patch, 'left': left, 'right': right,
        'base': review['base'], 'head': review['head'],
        'mode': mode, 'can_apply': review['can_apply']}

In [ ]:
r = GitRepo.at(mkrepo({'a.txt': 'one\n'}))
base = r._ask('rev-parse', 'HEAD')
commit(r.root, 'change a, add b', a__txt='two\n', b__txt='b\n')

one = r.review_file(base, 'HEAD', 'a.txt')
test_eq(one['file']['path'], 'a.txt')
test_eq((one['left'], one['right']), ('one\n', 'two\n'))
assert '+two' in one['patch']

test_eq(r._ref_file(base, 'a.txt'), 'one\n')
test_eq(r._ref_file(base, 'nothere.txt'), '')           # a file that was not there reads as empty
test_eq(r._ref_file('', 'a.txt'), '')
test_eq(r.review_file(base, 'HEAD', 'b.txt')['left'], '')
test_fail(lambda: r.review_file(base, 'HEAD', 'readme.md'), contains='not changed')

## Reviewing one commit

The same shape, for a commit against its parent. A root commit has no parent, so it is compared
against the empty tree.

In [ ]:
#| export
@patch
def commit_review(self:GitRepo, ref):
    oid = self._resolve_ref(ref)
    meta = self.commit_detail(oid)
    parent = meta['parents'][0] if meta['parents'] else self.run('mktree', input='').strip()
    status = self._diff_files(parent, oid)
    return {k: v for k, v in meta.items() if k != 'patch'} | {
        'base': parent, 'head': oid, 'files': status, 'summary': _summarise_diff(status)}

@patch
def commit_file(self:GitRepo, ref, path, context=3):
    detail = self.commit_review(ref)
    known = first(x for x in detail['files'] if x['path'] == path)
    if known is None: raise GitError(f'{path} is not changed by this commit')
    left, right, patch = self._file_pair(path, detail['base'], detail['head'],
        detail['base'][:8], detail['head'][:8], context, known.get('old_path'))
    return {'file': known, 'patch': patch, 'left': left, 'right': right,
        'base': detail['base'], 'head': detail['head']}

In [ ]:
r = GitRepo.at(mkrepo({'a.txt': 'one\n'}))
commit(r.root, 'change a, add b', a__txt='two\n', b__txt='b\n')

d = r.commit_review('HEAD')
test_eq(d['message'], 'change a, add b')
test_eq(sorted(f['path'] for f in d['files']), ['a.txt', 'b.txt'])
test_eq(d['summary']['files'], 2)
assert 'patch' not in d                                 # the per-file records replace it
test_eq(r.commit_file('HEAD', 'a.txt')['right'], 'two\n')
test_eq(r.commit_file('HEAD', 'a.txt')['left'], 'one\n')
test_fail(lambda: r.commit_file('HEAD', 'nothere.txt'), contains='not changed by this commit')

root = r.commit_review(r.history()[-1]['oid'])          # the first commit, against the empty tree
test_eq([f['path'] for f in root['files']], ['a.txt'])
test_eq(r.commit_file(r.history()[-1]['oid'], 'a.txt')['left'], '')

## Previewing integration

`merge_preview` and `rebase_preview` report what each operation would do without changing the repository. `compare` returns the complete snapshot difference between two refs.

In [ ]:
#| export
@patch
def merge_preview(self:GitRepo, incoming):
    current = self.run('branch', '--show-current').strip() or 'HEAD'
    current_oid, incoming_oid = self._resolve_ref('HEAD'), self._resolve_ref(incoming)
    merge_base = self._ask('merge-base', current_oid, incoming_oid)
    if not merge_base:
        raise GitError(f'{current} and {incoming} do not share a merge base')
    if current_oid == incoming_oid: relation = 'identical'
    elif merge_base == current_oid: relation = 'fast-forward'
    elif merge_base == incoming_oid: relation = 'already-merged'
    else: relation = 'diverged'
    conflicts, likely = self._rehearse(current_oid, incoming_oid)
    return ({'current': current, 'incoming': str(incoming), 'relation': relation}
        | self._in_the_way()
        | {'merge_base': merge_base, 'conflicts': conflicts, 'conflict_likely': likely,
            'review': self.review(current, incoming, 'review')})

@patch
def rebase_preview(self:GitRepo, onto):
    "Describe replaying current-only commits on `onto` before mutating history."
    current = self.run('branch', '--show-current').strip()
    if not current: raise GitError('cannot rebase a detached HEAD')
    current_oid, onto_oid = self._resolve_ref('HEAD'), self._resolve_ref(onto)
    base = self._ask('merge-base', current_oid, onto_oid)
    if not base: raise GitError(f'{current} and {onto} do not share a merge base')
    conflicts, likely = self._rehearse(onto_oid, current_oid)
    return ({'current': current, 'onto': str(onto)}
        | self._in_the_way()
        | {'merge_base': base, 'commits': self.history(limit=250, ref=f'{onto_oid}..{current_oid}'),
            'conflicts': conflicts, 'conflict_likely': likely,
            'already_based': base == onto_oid, 'review': self.review(onto, current, 'review')})

@patch
def compare(self:GitRepo, left, right):
    "Compare the complete snapshots at two refs, from `left` to `right`."
    review = self.review(left, right, 'snapshot')
    patch = self.run('diff', '--no-ext-diff', '--stat', '--patch',
        review['base_oid'], review['head_oid'], '--')
    return {'left': left, 'right': right, 'left_only': review['base_only'],
        'right_only': review['head_only'],
        'changed_files': review['summary']['files'],
        'merge_base': review['merge_base'], 'patch': patch}

In [ ]:
r = GitRepo.at(mkrepo({'a.txt': 'a\n'}))
sh(r.root, 'branch', 'other')
commit(r.root, 'ours', a__txt='ours\n')
r.checkout('other')
commit(r.root, 'theirs', b__txt='b\n')
head = r._ask('rev-parse', 'HEAD')

prev = r.merge_preview('main')
test_eq((prev['conflicts'], prev['conflict_likely']), ([], False))
test_eq(r._ask('rev-parse', 'HEAD'), head)              # rehearsing moved nothing
test_eq(r.rebase_preview('main')['conflicts'], [])
test_eq(r.rebase_preview('main')['already_based'], False)
cmp = r.compare('main', 'other')                        # whole trees, not the merge base
test_eq(cmp['changed_files'], 2)
test_eq((cmp['left_only'], cmp['right_only']), (1, 1))
assert 'b.txt' in cmp['patch'] and 'a.txt' in cmp['patch']

c = GitRepo.at(mkrepo({'shared.txt': 'base\n'}))
sh(c.root, 'branch', 'other')
commit(c.root, 'ours', shared__txt='ours\n')
c.checkout('other')
commit(c.root, 'theirs', shared__txt='theirs\n')
test_eq(c.merge_preview('main')['conflicts'], ['shared.txt'])
test_eq(c.merge_preview('main')['conflict_likely'], True)
test_eq(c._ask('rev-parse', 'HEAD'), c._ask('rev-parse', 'other'))

## Resolving one conflict

`conflict_versions` returns the base, ours, theirs, and working-tree versions of a conflicted file. `resolve_conflict` writes the selected version or caller-supplied text, then stages the file.

During a merge, `ours` is the checked-out branch and `theirs` is the incoming branch.

In [ ]:
#| export
@patch
def conflict_versions(self:GitRepo, path):
    change = first(row for row in self.changes() if row['path'] == path)
    if not change or not change['conflicted']: raise GitError(f'{path} is not conflicted')
    def stage(number):
        r = _run(self.root, 'show', f':{number}:{path}', check=False)
        return r.stdout if r.returncode == 0 else ''
    return {'path': path, 'base': stage(1), 'ours': stage(2), 'theirs': stage(3),
        'worktree': self._version(path, 'worktree'), 'status': change}

@patch
def resolve_conflict(self:GitRepo, path, choice, content=None):
    "Settle one conflicted file and stage it as resolved."
    if choice not in {'ours', 'theirs', 'worktree', 'resolved'}:
        raise GitError('conflict choice must be ours, theirs, worktree, or resolved')
    if choice == 'resolved':
        if content is None:
            raise GitError('a resolved file needs its resolved contents')
        self.write_worktree(path, content)
    elif choice != 'worktree':
        self._mutate('checkout', f'--{choice}', '--', path)
    self.stage([path])
    remaining = [c['path'] for c in self._changes_uncached() if c['conflicted']]
    return {'path': path, 'choice': choice, 'conflicted': remaining,
        'operation': self._operation(),
        'summary': f'{plural(len(remaining), "file")} still conflicted' if remaining else
            'every conflict is resolved -- commit to finish the merge'}

In [ ]:
r = GitRepo.at(mkrepo({'shared.txt': 'base\n'}))
sh(r.root, 'branch', 'other')
commit(r.root, 'ours', shared__txt='ours\n')
r.checkout('other')
commit(r.root, 'theirs', shared__txt='theirs\n')
r.merge('main')

v = r.conflict_versions('shared.txt')
test_eq((v['ours'], v['theirs'], v['base']), ('theirs\n', 'ours\n', 'base\n'))

out = r.resolve_conflict('shared.txt', 'ours')
test_eq((r.root/'shared.txt').read_text(), 'theirs\n')
test_eq(out['conflicted'], [])
test_eq(out['operation']['active'], 'merge')            # resolved, but not yet committed
assert 'every conflict is resolved' in out['summary']
r.operation_action('abort')

r.merge('main')
r.resolve_conflict('shared.txt', 'theirs')
test_eq((r.root/'shared.txt').read_text(), 'ours\n')
r.operation_action('abort')

r.merge('main')
r.resolve_conflict('shared.txt', 'resolved', 'decided by hand\n')
test_eq((r.root/'shared.txt').read_text(), 'decided by hand\n')
r.operation_action('abort')

r.merge('main')
test_fail(lambda: r.resolve_conflict('shared.txt', 'nonsense'), contains='conflict choice must be')
test_fail(lambda: r.resolve_conflict('shared.txt', 'resolved'), contains='resolved contents')
test_fail(lambda: r.conflict_versions('readme.md'), contains='is not conflicted')
r.operation_action('abort')

{'op': 'merge --abort',
 'message': '',
 'note': '',
 'head': '3532f46',
 'head_before': '3532f467f',
 'moved': False,
 'branch': 'other',
 'staged': [],
 'conflicted': [],
 'operation': {'active': '',
  'can_continue': False,
  'can_skip': False,
  'can_abort': False},
 'kept_work': False,
 'undo': '1a0426e3e13',
 'undoes': 'merge --abort on other, 1 uncommitted file(s) kept',
 'summary': 'merge --abort changed nothing'}

## Planning conflict resolution

`conflict_plan` describes unresolved files and marker blocks. `resolve_safe_conflicts` removes only blocks whose two sides are identical. Different sides remain for manual review.

In [ ]:
#| export
@patch
def conflict_plan(self:GitRepo):
    "Unresolved files with deterministic marker-level resolution suggestions."
    rows = []
    for change in self.changes():
        if not change['conflicted']: continue
        path, versions = change['path'], self.conflict_versions(change['path'])
        worktree = versions['worktree']
        if '\x00' in worktree:
            rows.append({'path': path, 'kind': 'binary', 'blocks': 0, 'safe': 0,
                         'manual': 1, 'reason': 'binary file; choose one complete side'})
            continue
        blocks = _conflict_blocks(worktree)
        safe = sum(1 for b in blocks if b['ours'] == b['theirs'])
        rows.append({'path': path, 'kind': 'text', 'blocks': len(blocks), 'safe': safe,
                     'manual': len(blocks) - safe,
                     'reason': 'identical sides can be removed safely' if safe else 'manual review required'})
    return {'operation': self._operation(), 'files': rows,
            'summary': {'files': len(rows), 'blocks': sum(x['blocks'] for x in rows),
                        'safe': sum(x['safe'] for x in rows), 'manual': sum(x['manual'] for x in rows)}}

@patch
def resolve_safe_conflicts(self:GitRepo, paths=()):
    "Remove only conflict markers whose ours and theirs text is identical, then stage."
    wanted = set(paths or [x['path'] for x in self.conflict_plan()['files']])
    changed = []
    for path in wanted:
        versions = self.conflict_versions(path)
        text, blocks = versions['worktree'], _conflict_blocks(versions['worktree'])
        if not blocks: continue
        out, at, safe = [], 0, 0
        for block in blocks:
            out.append(text[at:block['start']])
            if block['ours'] == block['theirs']:
                out.append(block['ours']); safe += 1
            else: out.append(text[block['start']:block['end']])
            at = block['end']
        out.append(text[at:])
        if not safe: continue
        self.write_worktree(path, ''.join(out))
        if safe == len(blocks): self.stage([path])
        changed.append({'path': path, 'safe': safe, 'remaining': len(blocks) - safe})
    return {'resolved': changed, 'plan': self.conflict_plan()}

In [ ]:
r = GitRepo.at(mkrepo({'shared.txt': 'base\n'}))
sh(r.root, 'branch', 'other')
commit(r.root, 'ours', shared__txt='ours\n')
r.checkout('other')
commit(r.root, 'theirs', shared__txt='theirs\n')
r.merge('main')

plan = r.conflict_plan()
test_eq([f['path'] for f in plan['files']], ['shared.txt'])
test_eq(plan['operation']['active'], 'merge')
test_eq(plan['files'][0]['kind'], 'text')
test_eq((plan['files'][0]['blocks'], plan['files'][0]['safe']), (1, 0))
test_eq(plan['summary'], {'files': 1, 'blocks': 1, 'safe': 0, 'manual': 1})

# Two sides that genuinely differ are nobody's to resolve automatically, so nothing is taken.
test_eq(r.resolve_safe_conflicts()['resolved'], [])
test_eq([f['path'] for f in r.resolve_safe_conflicts()['plan']['files']], ['shared.txt'])

# A block whose two sides say the same thing is safe, and taking it stages the file.
s = GitRepo.at(mkrepo({'both.txt': 'a\nb\nc\n'}))
sh(s.root, 'branch', 'other')
commit(s.root, 'ours', both__txt='a\nb\nOURS\n')
s.checkout('other')
commit(s.root, 'theirs', both__txt='a\nb\nTHEIRS\n')
s.merge('main')
marked = s._version('both.txt', 'worktree')
block, = _conflict_blocks(marked)
s.write_worktree('both.txt', marked[:block['start']] + '<<<<<<< HEAD\nsame\n=======\nsame\n'
                 '>>>>>>> main\n' + marked[block['end']:])
done = s.resolve_safe_conflicts()
test_eq([x['path'] for x in done['resolved']], ['both.txt'])
test_eq((done['resolved'][0]['safe'], done['resolved'][0]['remaining']), (1, 0))
test_eq(done['plan']['files'], [])
test_eq((s.root/'both.txt').read_text(), 'a\nb\nsame\n')

r.operation_action('abort')
test_eq(r.conflict_plan()['files'], [])                 # nothing in progress, nothing to decide

## Blame

One `blame --line-porcelain` per range, as one record per line.

In [ ]:
#| export
@patch
def blame(self:GitRepo, path, start=1, end=0):
    args = ['blame', '--line-porcelain']
    if end: args += ['-L', f'{max(1, int(start))},{max(int(start), int(end))}']
    args += ['--', path]
    rows, current = [], None
    for line in self.run(*args).splitlines():
        if re.match(r'^[0-9a-f^]{40} ', line):
            oid, original, final, count = (line.split() + ['1'])[:4]
            current = {'oid': oid.lstrip('^'), 'original_line': int(original),
                'line': int(final), 'count': int(count), 'author': '',
                'timestamp': 0, 'summary': '', 'text': ''}
        elif current is None: continue
        elif line.startswith('author '): current['author'] = line[7:]
        elif line.startswith('author-time '): current['timestamp'] = int(line[12:] or 0)
        elif line.startswith('summary '): current['summary'] = line[8:]
        elif line.startswith('\t'):
            current['text'] = line[1:]
            rows.append(current)
            current = None
    return rows

In [ ]:
r = GitRepo.at(mkrepo({'a.txt': 'one\ntwo\n'}))
rows = r.blame('a.txt')
test_eq(len(rows), 2)
test_eq(rows[0]['author'], 'Repo tests')
test_eq(rows[0]['line'], 1)
test_eq(rows[0]['summary'], 'initial')
test_eq(r.blame('a.txt', 2, 2)[0]['line'], 2)
test_eq(len(r.blame('a.txt', 1, 1)), 1)

commit(r.root, 'change the second line', a__txt='one\nTWO\n')
test_eq([x['summary'] for x in r.blame('a.txt')], ['initial', 'change the second line'])

## Worktrees and submodules

The two ways one checkout is really several. Both are reads, plus the two commands that add and
remove a worktree.

In [ ]:
#| export
@patch
def worktrees(self:GitRepo):
    rows, current = [], None
    for line in self.run('worktree', 'list', '--porcelain').splitlines() + ['']:
        if line.startswith('worktree '):
            current = {'path': line[9:], 'head': '', 'branch': '', 'bare': False, 'detached': False, 'locked': False}
        elif not line and current:
            rows.append(current)
            current = None
        elif current is not None:
            key, _, value = line.partition(' ')
            if key in {'bare', 'detached', 'locked'}: current[key] = True
            elif key == 'branch': current[key] = value.removeprefix('refs/heads/')
            elif key == 'HEAD': current['head'] = value
    return rows

@patch
def add_worktree(self:GitRepo, path, branch='', create=False):
    args = ['worktree', 'add']
    if create and branch: args += ['-b', branch]
    args += [path]
    if branch and not create: args.append(branch)
    return self._mutate(*args).strip()

@patch
def remove_worktree(self:GitRepo, path, force=False):
    return self._mutate('worktree', 'remove', *(['--force'] if force else []), path).strip()

@patch
def submodules(self:GitRepo):
    r = _run(self.root, 'submodule', 'status', '--recursive', check=False)
    if r.returncode and not (self.root/'.gitmodules').exists(): return []
    rows = []
    for line in r.stdout.splitlines():
        state, body = line[:1], line[1:].strip()
        oid, _, tail = body.partition(' ')
        path, _, description = tail.partition(' ')
        rows.append({'path': path, 'oid': oid, 'state': state,
            'description': description.strip('()')})
    return rows

In [ ]:
r = GitRepo.at(mkrepo())
trees = r.worktrees()
test_eq(len(trees), 1)
test_eq((trees[0]['path'], trees[0]['branch']), (str(r.root), 'main'))
test_eq((trees[0]['bare'], trees[0]['detached']), (False, False))

side = Path(tempfile.mkdtemp())/'side'; _tmp.append(str(side.parent))
r.add_worktree(side, 'sidebranch', create=True)
test_eq(sorted(w['branch'] for w in r.worktrees()), ['main', 'sidebranch'])
r.remove_worktree(side, force=True)
test_eq(len(r.worktrees()), 1)

test_eq(r.submodules(), [])

## The diff itself

`diff` is `git diff` with the pane's arguments: staged or not, or any two refs, with the context a
pane can usefully show.

In [ ]:
#| export
@patch
def diff(self:GitRepo, path='', staged=False, left='', right='', context=3):
    args = ['diff', '--no-ext-diff', _unified_arg(context)]
    if staged: args.append('--cached')
    if left and right: args.append(f'{left}...{right}')
    elif left: args.append(left)
    if path: args += ['--', path]
    return self.run(*args)

In [ ]:
r = GitRepo.at(mkrepo({'a.txt': 'one\n'}))
write(r.root, 'a.txt', 'two\n')
assert '+two' in r.diff('a.txt')
test_eq(r.diff('a.txt', staged=True), '')
r.stage(['a.txt'])
assert '+two' in r.diff('a.txt', staged=True)
test_eq(r.diff('a.txt'), '')                            # staged, so nothing left unstaged
test_eq(r._version('a.txt', 'index'), 'two\n')          # ...which is where the index now is
test_eq(r._version('a.txt', 'head'), 'one\n')

base = r._ask('rev-parse', 'HEAD')
commit(r.root, 'second', a__txt='three\n')
assert '+three' in r.diff('a.txt', left=base, right='HEAD')

## Rendering notebooks and LFS pointers

Notebook diffs compare flattened cell sources instead of notebook JSON. LFS pointers remain marked as binary content because their text describes another file rather than the file itself.

In [ ]:
#| export
LFS_MAGIC = 'version https://git-lfs.github.com/spec/v1'

def _notebook_content(raw):
    if not raw: return ''
    try: cells = (json.loads(raw) or {}).get('cells') or []
    except (TypeError, json.JSONDecodeError): return raw
    parts = []
    for i, cell in enumerate(cells, 1):
        source = cell.get('source') or ''
        if isinstance(source, list): source = ''.join(source)
        kind = cell.get('cell_type') or 'cell'
        parts.append(f'# %% {kind} · cell {i}\n{source.rstrip()}')
    return '\n\n'.join(parts) + ('\n' if parts else '')

def _lfs_pointer(raw):
    "The oid and size of an LFS pointer, or None. A pointer is text, so git diffs it as text."
    if not raw or not raw.startswith(LFS_MAGIC): return None
    fields = dict(l.split(' ', 1) for l in raw.strip().split('\n') if ' ' in l)
    return {'oid': fields.get('oid', ''), 'size': fields.get('size', '')}

@patch
def diff_view(self:GitRepo, path, staged=False):
    change = first(c for c in self.changes() if c['path'] == path)
    if change and change['untracked'] and not staged:
        raw = _unified('', self._version(path, 'worktree'), '/dev/null', f'b/{path}')
    else:
        raw = self.diff(path, staged)
    left = self._version(path, 'head' if staged else 'index')
    right = self._version(path, 'index' if staged else 'worktree')
    lfs = _lfs_pointer(left) or _lfs_pointer(right)
    binary = bool(lfs) or 'Binary files' in raw
    if binary:
        return {'text': raw, 'content': '', 'notebook': False, 'left': '', 'right': '',
                'binary': True, 'lfs': lfs}
    if not str(path).lower().endswith('.ipynb'):
        return {'text': raw, 'content': raw, 'notebook': False, 'left': left, 'right': right,
                'binary': False, 'lfs': None}
    left, right, content = self._side_by_side(path, left, right, 'before', 'after', raw)
    return {'text': raw, 'content': content, 'notebook': True, 'left': left, 'right': right,
            'binary': False, 'lfs': None}

In [ ]:
nb = json.dumps({'cells': [{'cell_type': 'code', 'source': ['x = 1\n']},
                           {'cell_type': 'markdown', 'source': '# heading'}]})
flat = _notebook_content(nb)
assert 'x = 1' in flat and '# heading' in flat and '# %% code' in flat
test_eq(_notebook_content(''), '')
test_eq(_notebook_content('not json'), 'not json')      # not a notebook, so left alone

test_eq(_lfs_pointer(LFS_MAGIC + '\noid sha256:abc\nsize 12\n'), {'oid': 'sha256:abc', 'size': '12'})
test_eq(_lfs_pointer('ordinary text'), None)
test_eq(_lfs_pointer(''), None)

r = GitRepo.at(mkrepo({'a.txt': 'one\n'}))
write(r.root, 'a.txt', 'two\n')
view = r.diff_view('a.txt')
test_eq((view['left'], view['right']), ('one\n', 'two\n'))
test_eq((view['binary'], view['lfs'], view['notebook']), (False, None, False))

write(r.root, 'new.txt', 'fresh\n')
fresh = r.diff_view('new.txt')
test_eq((fresh['left'], fresh['right']), ('', 'fresh\n'))   # untracked: nothing on the left
assert '+fresh' in fresh['text']

nb = GitRepo.at(mkrepo({'n.ipynb': json.dumps({'cells': [{'id': 'a', 'cell_type': 'code',
                                                          'source': 'x = 1\n'}]}, indent=1)}))
write(nb.root, 'n.ipynb', json.dumps({'cells': [{'id': 'a', 'cell_type': 'code',
                                                 'source': 'x = 2\n'}]}, indent=1))
view = nb.diff_view('n.ipynb')
test_eq(view['notebook'], True)
test_eq((view['left'], view['right']), ('# %% code \u00b7 cell 1\nx = 1\n', '# %% code \u00b7 cell 1\nx = 2\n'))

## Notebooks

A notebook that changed by one cell should not read as a file that changed entirely. `_cells` keys a
notebook's sources by cell id so that the comparison is cell against cell, and `_line_ranges` says
which lines inside a changed cell moved.

In [ ]:
#| export
def _cells(raw):
    "id -> source for one notebook's cells, in order. A cell with no id is keyed by position."
    try: cells = (json.loads(raw) or {}).get('cells') or []
    except (TypeError, json.JSONDecodeError): return {}
    out = {}
    for i, c in enumerate(cells):
        src = c.get('source') or ''
        if isinstance(src, list): src = ''.join(src)
        out[str(c.get('id') or f'#{i}')] = src
    return out

def _line_ranges(before, after):
    "Lines of `after` that differ from `before`, 1-based, in `file_changes`'s shape."
    old, new = before.splitlines(), after.splitlines()
    out = []
    for tag, _, _, b1, b2 in difflib.SequenceMatcher(None, old, new, autojunk=False).get_opcodes():
        if tag == 'equal': continue
        if tag == 'delete':
            at = max(1, min(b1 + 1, len(new) or 1))
            out.append({'from': at, 'to': at, 'kind': 'deleted'})
        else:
            out.append({'from': b1 + 1, 'to': max(b1 + 1, b2), 'kind': 'added' if tag == 'insert' else 'modified'})
    return out

@patch
def notebook_changes(self:GitRepo, path):
    "Which cells of a notebook differ from HEAD, and which lines within them."
    out = {'path': path, 'changed': False, 'cells': {}, 'lines': {}}
    if not str(path).lower().endswith('.ipynb'): return out
    changed = first(c for c in self.changes() if c['path'] == path)
    if changed is None: return out
    now = _cells(self._version(path, 'worktree'))
    was = {} if changed['untracked'] else _cells(self._version(path, 'head'))
    for cid, src in now.items():
        if was.get(cid) == src: continue
        out['cells'][cid] = 'added' if cid not in was else 'modified'
        out['lines'][cid] = _line_ranges(was.get(cid, ''), src)
    out['changed'] = bool(out['cells'])
    return out

In [ ]:
nb = json.dumps({'cells': [{'id': 'a', 'cell_type': 'code', 'source': 'x = 1\n'},
                           {'cell_type': 'code', 'source': ['y', ' = 2\n']}]})
test_eq(_cells(nb), {'a': 'x = 1\n', '#1': 'y = 2\n'})   # a cell with no id is keyed by position
test_eq(_cells('not json'), {})

test_eq(_line_ranges('a\n', 'a\n'), [])
test_eq(_line_ranges('a\n', 'a\nb\n'), [{'from': 2, 'to': 2, 'kind': 'added'}])
test_eq(_line_ranges('a\nb\n', 'a\nB\n'), [{'from': 2, 'to': 2, 'kind': 'modified'}])
# a deletion has no line of its own in `after`, so it is marked at the line that took its place
test_eq(_line_ranges('a\nb\n', 'a\n'), [{'from': 1, 'to': 1, 'kind': 'deleted'}])
test_eq(_line_ranges('a\n', ''), [{'from': 1, 'to': 1, 'kind': 'deleted'}])
test_eq(_line_ranges('a\nb\nc\n', 'a\nb\nc\nd\ne\n'), [{'from': 4, 'to': 5, 'kind': 'added'}])

before = json.dumps({'cells': [{'id': 'a', 'cell_type': 'code', 'source': 'x = 1\n'},
                               {'id': 'b', 'cell_type': 'code', 'source': 'y = 2\n'}]}, indent=1)
after = json.dumps({'cells': [{'id': 'a', 'cell_type': 'code', 'source': 'x = 99\n'},
                              {'id': 'b', 'cell_type': 'code', 'source': 'y = 2\n'}]}, indent=1)
r = GitRepo.at(mkrepo({'n.ipynb': before}))
test_eq(r.notebook_changes('n.ipynb')['changed'], False)
write(r.root, 'n.ipynb', after)
out = r.notebook_changes('n.ipynb')
test_eq(out['changed'], True)
test_eq(sorted(out['cells']), ['a'])                    # only the cell that moved
test_eq(out['lines']['a'], [{'from': 1, 'to': 1, 'kind': 'modified'}])
test_eq(r.notebook_changes('a.txt')['changed'], False)   # not a notebook, so nothing to say

## Reading a patch

`_patch_hunks` turns a unified diff into the spans a gutter draws, one record per `@@` header.

In [ ]:
#| export
def _patch_hunks(diff):
    "Every hunk of one file's unified diff, as a patch that can be applied on its own."
    lines = str(diff or '').split('\n')
    start = first(i for i, x in enumerate(lines) if x.startswith('@@'))
    if start is None: return []
    header, chunks, current = '\n'.join(lines[:start]) + '\n', [], []
    for line in lines[start:]:
        if line.startswith('@@') and current:
            chunks.append(current)
            current = []
        current.append(line)
    if current: chunks.append(current)
    out = []
    for chunk in chunks:
        m = re.match(r'^@@ -\d+(?:,\d+)? \+(\d+)(?:,\d+)? @@', chunk[0])
        if not m: continue
        at, old, new, adds, anchor = int(m.group(1)), [], [], [], None
        for line in chunk[1:]:
            if line.startswith('\\'): continue
            if line.startswith('-'):
                old.append(line[1:])
                if anchor is None: anchor = max(1, at - 1)
            elif line.startswith('+'):
                new.append(line[1:])
                adds.append(at)
                at += 1
            else: at += 1
        if not old and not new: continue
        span = (min(adds), max(adds)) if adds else (anchor, anchor)
        out.append({'index': len(out), 'from': span[0], 'to': span[1],
            'kind': 'added' if not old else 'deleted' if not new else 'modified',
            'old': ''.join(x + '\n' for x in old), 'new': ''.join(x + '\n' for x in new),
            'patch': header + '\n'.join(chunk).rstrip('\n') + '\n'})
    return out

In [ ]:
diff = ('diff --git a/a.txt b/a.txt\n--- a/a.txt\n+++ b/a.txt\n'
        '@@ -1,2 +1,3 @@\n a\n-b\n+B\n+c\n')
hunks = _patch_hunks(diff)
test_eq(len(hunks), 1)
test_eq((hunks[0]['from'], hunks[0]['to']), (2, 3))
test_eq(_patch_hunks(''), [])
test_eq(_patch_hunks('not a diff\n'), [])

two = _patch_hunks('@@ -1 +1 @@\n-a\n+A\n@@ -10,2 +10,2 @@\n-b\n+B\n')
test_eq([(h['from'], h['to'], h['kind']) for h in two], [(1, 1, 'modified'), (10, 10, 'modified')])
test_eq(two[0]['old'], 'a\n')
test_eq(two[0]['new'], 'A\n')
# A hunk that only deletes has no added line to sit on, so it anchors above the deletion.
test_eq([(h['from'], h['to'], h['kind']) for h in _patch_hunks('@@ -5,2 +4,1 @@\n a\n-b\n')],
        [(4, 4, 'deleted')])

## What the gutter draws

`file_changes` is one file's current diff, its hunks, and the status record that says how the file
got that way.

In [ ]:
#| export
@patch
def file_changes(self:GitRepo, path):
    changed = first(c for c in self.changes() if c['path'] == path)
    if changed is None:
        return {'path': path, 'changed': False, 'ranges': [], 'hunks': [], 'diff': ''}
    if changed['untracked']:
        lines = max(1, len(self._version(path, 'worktree').splitlines()))
        return {'path': path, 'changed': True,
            'ranges': [{'from': 1, 'to': lines, 'kind': 'added'}],
            'hunks': [], 'diff': '', 'status': changed}
    r = _run(self.root, 'diff', '--no-ext-diff', '--unified=3', 'HEAD', '--', path, check=False)
    exact = _run(self.root, 'diff', '--no-ext-diff', '--unified=0', 'HEAD', '--', path, check=False)
    if r.returncode:
        r = _run(self.root, 'diff', '--no-ext-diff', '--unified=3', '--', path)
        exact = _run(self.root, 'diff', '--no-ext-diff', '--unified=0', '--', path)
    ranges = []
    for m in re.finditer(r'^@@ -\d+(?:,(\d+))? \+(\d+)(?:,(\d+))? @@', exact.stdout, re.M):
        old_n, start, new_n = int(m.group(1) or 1), int(m.group(2)), int(m.group(3) or 1)
        kind = 'added' if old_n == 0 else 'deleted' if new_n == 0 else 'modified'
        start = max(1, start)
        end = max(start, start + max(1, new_n) - 1)
        ranges.append({'from': start, 'to': end, 'kind': kind})
    return {'path': path, 'changed': True, 'ranges': ranges,
        'hunks': _patch_hunks(r.stdout), 'diff': r.stdout, 'status': changed}

In [ ]:
r = GitRepo.at(mkrepo({'a.txt': 'a\nb\n'}))
test_eq(r.file_changes('a.txt')['changed'], False)
test_eq(r.file_changes('missing.txt'), {'path': 'missing.txt', 'changed': False,
                                        'ranges': [], 'hunks': [], 'diff': ''})

write(r.root, 'a.txt', 'a\nB\nc\n')
out = r.file_changes('a.txt')
test_eq(len(out['hunks']), 1)
test_eq(out['ranges'], [{'from': 2, 'to': 3, 'kind': 'modified'}])
assert out['diff'] and out['status']

write(r.root, 'new.txt', 'fresh\n')
untracked = r.file_changes('new.txt')
test_eq(untracked['hunks'], [])                         # nothing to diff against
test_eq(untracked['ranges'], [{'from': 1, 'to': 1, 'kind': 'added'}])

## Replaying a branch in memory

A rebase rewrites history, so the only honest way to say what one would do is to do it somewhere
that does not count. `_replay` merges each commit onto the last with `merge-tree` and wraps the
result back into a commit with `commit-tree`. Both write loose objects and touch no ref, index or
worktree, so the rehearsal is free and leaves nothing behind.

In [ ]:
#| export
SYNC_OPS = ('fast-forward', 'merge', 'rebase', 'reset')

REMOTE_OPS = ('fetch', 'pull', 'push')

STATE_KEYS = ('root', 'branch', 'upstream', 'ahead', 'behind', 'clean', 'branches', 'changes')

def _short_commits(rows, n=20):
    "Commits as one line each."
    return [f'{r["short"]} {shorten(r["subject"], 72)}' for r in rows[:n]]

def _said(out):
    "What a mutation reported, whichever shape it reported it in."
    if isinstance(out, dict): return out.get('summary') or out.get('message') or ''
    return str(out or '').strip()

@patch
def _upstream(self: GitRepo, name=''):
    "The named upstream, or the one this branch tracks."
    up = str(name or '').strip() or self._ask('rev-parse', '--abbrev-ref', '@{upstream}')
    if not up: raise GitError('this branch tracks nothing -- push it, or name an upstream')
    return up

@patch
def _replay(self: GitRepo, onto, commits):
    "Replay `commits` onto `onto` in memory, stopping where a rebase would."
    head = onto
    for n, row in enumerate(commits):
        tree, files, clashed = self._merge_tree(head, row['oid'], self._ask('rev-parse', f'{row["oid"]}^'))
        if clashed: return {'oid': row['short'], 'subject': row['subject']}, n, files
        # Each result wrapped back into a commit. The next step replays onto what the last left.
        head = _run(self.root, '-c', 'user.name=leela', '-c', 'user.email=leela@localhost',
                    'commit-tree', tree, '-p', head, '-m', row['subject']).stdout.strip() or head
    return None, len(commits), []

In [ ]:
test_eq(_short_commits([{'short': 'abc1234', 'subject': 'a subject'}]), ['abc1234 a subject'])
test_eq(_short_commits([{'short': 'a', 'subject': 'x'}] * 30), ['a x'] * 20)   # capped for a summary
test_eq(_said({'summary': 's'}), 's')
test_eq(_said({'message': 'm'}), 'm')
test_eq(_said(' text '), 'text')
test_eq(_said(None), '')
test_eq(sorted(SYNC_OPS), ['fast-forward', 'merge', 'rebase', 'reset'])
test_eq(REMOTE_OPS, ('fetch', 'pull', 'push'))

r = GitRepo.at(mkrepo({'a.txt': 'base\n'}))
test_fail(lambda: r._upstream(), contains='tracks nothing')
sh(r.root, 'branch', 'other')
commit(r.root, 'ours', b__txt='b\n')
onto = r._ask('rev-parse', 'HEAD')
r.checkout('other')
commit(r.root, 'theirs', c__txt='c\n')
head = r._ask('rev-parse', 'HEAD')

stops, replayed, files = r._replay(onto, list(reversed(r.history(ref=f'{onto}..HEAD'))))
test_eq((stops, replayed, files), (None, 1, []))        # replays cleanly
test_eq(r._ask('rev-parse', 'HEAD'), head)              # ...and moved nothing

c = GitRepo.at(mkrepo({'shared.txt': 'base\n'}))
sh(c.root, 'branch', 'other')
commit(c.root, 'ours', shared__txt='ours\n')
onto = c._ask('rev-parse', 'HEAD')
c.checkout('other')
commit(c.root, 'theirs', shared__txt='theirs\n')
stops, replayed, files = c._replay(onto, list(reversed(c.history(ref=f'{onto}..HEAD'))))
test_eq(stops['subject'], 'theirs')                     # the commit a rebase would stop at
test_eq((replayed, files), (0, ['shared.txt']))

## Choosing a synchronization strategy

`divergence` compares the current branch with its upstream and rehearses fast-forward, merge, rebase, and reset options. It recommends a fast-forward when there are no local commits. It recommends rebase when local commits replay cleanly on a clean tree. Otherwise it recommends merge.

In [ ]:
#| export
@patch
def divergence(self: GitRepo, upstream='', fetch=False):
    "You against your upstream, every way back rehearsed before any of them runs."
    if fetch: self.fetch()
    upstream = self._upstream(upstream)
    branch = self.run('branch', '--show-current').strip()
    ours_oid, theirs_oid = self._resolve_ref('HEAD'), self._resolve_ref(upstream)
    base = self._ask('merge-base', ours_oid, theirs_oid)
    if not base: raise GitError(f'{branch or "HEAD"} and {upstream} do not share a merge base')
    ours = self.history(limit=250, ref=f'{theirs_oid}..{ours_oid}')
    theirs = self.history(limit=250, ref=f'{ours_oid}..{theirs_oid}')
    ahead, behind = len(ours), len(theirs)
    relation = ('identical' if ours_oid == theirs_oid else 'behind' if not ahead else
                'ahead' if not behind else 'diverged')
    conflicts, likely = self._rehearse(ours_oid, theirs_oid)
    stops, replayed, replay_conflicts = self._replay(theirs_oid, list(reversed(ours)))
    blocked = self._in_the_way()
    options = [
        {'op': 'fast-forward', 'available': not ahead and bool(behind), 'destructive': False,
         'note': 'move straight onto the upstream' if not ahead else 'your own commits are in the way'},
        {'op': 'merge', 'available': bool(behind), 'destructive': False, 'conflicts': conflicts,
         'conflict_likely': likely, 'note': 'one merge commit, and any conflict resolved once'},
        {'op': 'rebase', 'available': bool(behind and ahead), 'destructive': True,
         'conflicts': replay_conflicts, 'conflict_likely': bool(stops), 'stops_at': stops,
         'replayed': replayed, 'note': 'a linear history, rewritten, and a conflict per commit'},
        {'op': 'reset', 'available': bool(behind), 'destructive': True,
         'note': f'throw away {plural(ahead, "commit")} and take the upstream as it is'}]
    recommended = ('' if relation in ('identical', 'ahead') else 'fast-forward' if not ahead else
                   'merge' if stops or not blocked['clean'] else 'rebase')
    return {'branch': branch, 'upstream': upstream, 'relation': relation, 'ahead': ahead,
            'behind': behind, 'merge_base': base, 'options': options, 'recommended': recommended,
            'ours': _short_commits(ours), 'theirs': _short_commits(theirs)} | blocked

In [ ]:
r = GitRepo.at(mkrepo({'a.txt': 'base\n'}))
bare = mkbare()
sh(r.root, 'remote', 'add', 'origin', str(bare))
sh(r.root, 'push', '-u', 'origin', 'main')
test_eq(r._upstream(), 'origin/main')

d = r.divergence()
test_eq(d['relation'], 'identical')
test_eq((d['ahead'], d['behind'], d['recommended']), (0, 0, ''))
test_eq(sorted(o['op'] for o in d['options']), sorted(SYNC_OPS))

other = Path(tempfile.mkdtemp()); _tmp.append(str(other))
o = GitRepo.at(clone(str(bare), other, 'other'))
sh(o.root, 'config', 'user.email', 'tests@example.com')
sh(o.root, 'config', 'user.name', 'Repo tests')
commit(o.root, 'upstream moved', b__txt='b\n')
o.push()

r.fetch()
test_eq(r.divergence()['relation'], 'behind')
test_eq(r.divergence()['recommended'], 'fast-forward')

commit(r.root, 'we moved', c__txt='c\n')
d = r.divergence()
test_eq((d['relation'], d['ahead'], d['behind']), ('diverged', 1, 1))
test_eq(d['recommended'], 'rebase')                     # nothing rehearsed a conflict
test_eq({x['op']: x['available'] for x in d['options']}['fast-forward'], False)
test_eq({x['op']: x for x in d['options']}['rebase']['stops_at'], None)
test_eq(d['ours'], ['%s we moved' % r._ask('rev-parse', '--short', 'HEAD')])

## Doing it

`sync` runs whichever way back was chosen, guarded like every other mutation, with the work set aside
where the operation needs a clean tree.

In [ ]:
#| export
@patch
def sync(self: GitRepo, how='rebase', upstream=''):
    "Reconcile with the upstream the way `divergence` rehearsed."
    if how not in SYNC_OPS: raise GitError(f'sync must be one of {", ".join(SYNC_OPS)}')
    ref = self._resolve_ref(self._upstream(upstream))
    args = {'fast-forward': ('merge', '--ff-only', ref), 'merge': ('merge', '--no-edit', ref),
            'rebase': ('rebase', ref), 'reset': ('reset', '--hard', ref)}[how]
    return self._guarded(how, lambda: self._attempt(*args), autostash=how in ('rebase', 'reset'))

In [ ]:
r = GitRepo.at(mkrepo({'a.txt': 'base\n'}))
bare = mkbare()
sh(r.root, 'remote', 'add', 'origin', str(bare))
sh(r.root, 'push', '-u', 'origin', 'main')
other = Path(tempfile.mkdtemp()); _tmp.append(str(other))
o = GitRepo.at(clone(str(bare), other, 'other'))
sh(o.root, 'config', 'user.email', 'tests@example.com')
sh(o.root, 'config', 'user.name', 'Repo tests')
commit(o.root, 'upstream moved', b__txt='b\n')
o.push()
commit(r.root, 'we moved', c__txt='c\n')
r.fetch()

test_fail(lambda: r.sync('sideways'), contains='sync must be one of')
test_eq(r.sync('rebase')['op'], 'rebase')
test_eq(r.divergence()['relation'], 'ahead')
r.push()
test_eq(r.divergence(fetch=True)['relation'], 'identical')

# ...and a divergence that would conflict recommends the merge instead.
o.pull()
commit(o.root, 'theirs', shared__txt='theirs\n')
o.push()
c = GitRepo.at(clone(str(bare), other, 'third'))
sh(c.root, 'config', 'user.email', 'tests@example.com')
sh(c.root, 'config', 'user.name', 'Repo tests')
sh(c.root, 'reset', '--hard', 'HEAD~1')
commit(c.root, 'ours', shared__txt='ours\n')
d = c.divergence()
test_eq(d['relation'], 'diverged')
test_eq(d['recommended'], 'merge')
test_eq({x['op']: x for x in d['options']}['rebase']['conflict_likely'], True)

## Ignore rules a folder needs

`git add` honours `.gitignore`, so a folder that has not got one yet stages `target/`, `dist/`, `__pycache__` and `node_modules` along with the work. `project_kinds` reads the marker files that say what kind of project a folder holds.

In [ ]:
#| export
#: Per project kind: the files that say a folder is one, a path that must end up ignored, and the
#: block to write when it does not.
IGNORE_KINDS = {
    'rust': dict(
        markers=('Cargo.toml',), probes=('target/debug/build',),
        lines=('# Rust', '/target/', '**/*.rs.bk')),
    'maturin': dict(
        markers=('pyproject.toml', 'Cargo.toml'), probes=('dist/x.whl',),
        lines=('# Wheels built from this crate', '/dist/', '*.so', '*.pyd')),
    'python': dict(
        markers=('pyproject.toml', 'setup.py', 'setup.cfg'), probes=('__pycache__/x.pyc', '.venv/x'),
        lines=('# Python', '__pycache__/', '*.py[cod]', '.venv/', 'venv/', '/build/',
               '*.egg-info/', '.pytest_cache/', '.ipynb_checkpoints/')),
    'node': dict(
        markers=('package.json',), probes=('node_modules/x',),
        lines=('# Node', 'node_modules/', '*.tsbuildinfo')),
}

#: `maturin` needs every marker; the rest need any one. A `pyproject.toml` beside a `Cargo.toml` is
#: a Python package built from Rust, and its `dist/` holds wheels rather than an sdist.
_ALL_MARKERS = ('maturin',)

def project_kinds(folder):
    "Which kinds of project this folder holds, by the files that declare one."
    folder = Path(folder)
    out = []
    for kind, spec in IGNORE_KINDS.items():
        hit = all if kind in _ALL_MARKERS else any
        if hit((folder/m).exists() for m in spec['markers']): out.append(kind)
    if 'maturin' in out and 'python' in out: out.remove('python')   # the same `dist/`, said twice
    return out

In [ ]:
d = Path(tempfile.mkdtemp()); _tmp.append(str(d))
test_eq(project_kinds(d), [])
(d/'package.json').write_text('{}')
test_eq(project_kinds(d), ['node'])
(d/'Cargo.toml').write_text('[package]\n')
test_eq(project_kinds(d), ['rust', 'node'])             # marker order is `IGNORE_KINDS` order
(d/'pyproject.toml').write_text('[project]\n')
project_kinds(d)                                        # a crate plus a pyproject is maturin, not python

Git is asked rather than guessed at. `missing_ignores` puts each kind's `probes` through `check-ignore`, so a `.gitignore` somebody wrote by hand in another shape still counts as sufficient and the repository is left alone.

In [ ]:
#| export
def _ignored(repo, path):
    "Whether git already ignores this path."
    try: rel = Path(path).resolve().relative_to(Path(repo.root).resolve())
    except ValueError: return False
    try: repo.run('check-ignore', '-q', '--no-index', str(rel))
    except (GitError, OSError): return False
    return True

def missing_ignores(repo, folder):
    "The `(kind, lines)` this folder needs and the repository does not already cover."
    folder = Path(folder)
    out = []
    for kind in project_kinds(folder):
        spec = IGNORE_KINDS[kind]
        if all(_ignored(repo, folder/p) for p in spec['probes']): continue
        out.append((kind, list(spec['lines'])))
    return out

In [ ]:
r = GitRepo.at(mkrepo())
(r.root/'crate').mkdir(); (r.root/'crate'/'Cargo.toml').write_text('[package]\n')
test_eq([k for k, _ in missing_ignores(r, r.root/'crate')], ['rust'])
test_eq(_ignored(r, r.root/'crate'/'target'/'debug'/'build'), False)
write(r.root, '.gitignore', 'crate/target/\n')
test_eq(_ignored(r, r.root/'crate'/'target'/'debug'/'build'), True)
test_eq(missing_ignores(r, r.root/'crate'), [])         # already covered, in a shape of its own
test_eq(_ignored(r, '/elsewhere'), False)               # outside the worktree is never ignored
(r.root/'pyproject.toml').write_text('[project]\n')
missing_ignores(r, r.root)

The rules go in the folder's own `.gitignore`, not the repository's. `/target/` at the root of a repository does not match `crate/target/`, so a crate in a subfolder written the other way would ignore nothing at all, and a crate carrying its own ignores survives being moved into another repository.

`added` is empty when the rules are already there, which is the common case on a second run and on a checkout somebody else set up.

In [ ]:
#| export
def prepare_ignores(repo, folder, write=True):
    "Give `folder` the ignore rules its project kinds need, and say what was added."
    missing = missing_ignores(repo, folder)
    wanted = [l for _, lines in missing for l in lines]
    target = Path(folder)/'.gitignore'
    added = list(wanted)
    if write and wanted:
        old = target.read_text(encoding='utf-8').splitlines() if target.exists() else []
        added = [l for l in wanted if l not in old]
        if added:
            head = old + ([''] if old and old[-1].strip() else [])
            target.write_text('\n'.join(head + added) + '\n', encoding='utf-8')
            invalidate(repo.root)
    return {'kinds': [k for k, _ in missing], 'added': added, 'path': str(target)}

In [ ]:
r = GitRepo.at(mkrepo())
(r.root/'web').mkdir(); (r.root/'web'/'package.json').write_text('{}')
dry = prepare_ignores(r, r.root/'web', write=False)
test_eq(dry['kinds'], ['node'])
test_eq((r.root/'web'/'.gitignore').exists(), False)    # write=False writes nothing

done = prepare_ignores(r, r.root/'web')
test_eq(done['added'], ['# Node', 'node_modules/', '*.tsbuildinfo'])
test_eq((r.root/'web'/'.gitignore').read_text(), '# Node\nnode_modules/\n*.tsbuildinfo\n')
test_eq(prepare_ignores(r, r.root/'web')['added'], [])  # the second run has nothing to add
(r.root/'web'/'node_modules').mkdir(); (r.root/'web'/'node_modules'/'x').write_text('x')
test_eq(_ignored(r, r.root/'web'/'node_modules'/'x'), True)  # the folder's own rules now cover it
done

In [ ]:
#| hide
r = GitRepo.at(mkrepo())
(r.root/'lib').mkdir(); (r.root/'lib'/'.gitignore').write_text('*.log')   # no trailing newline
(r.root/'lib'/'package.json').write_text('{}')
prepare_ignores(r, r.root/'lib')
test_eq((r.root/'lib'/'.gitignore').read_text(), '*.log\n\n# Node\nnode_modules/\n*.tsbuildinfo\n')
test_eq(prepare_ignores(r, r.root/'lib')['added'], [])
test_eq(prepare_ignores(r, r.root/'nope'), {'kinds': [], 'added': [], 'path': str(r.root/'nope'/'.gitignore')})

In [ ]:
#| hide
# The three helpers an embedder had to reach past the underscore for. Public, exported, and each
# one still doing what the private name did.
import gheasy.repo as _r
for n in ('invalidate', 'plural', 'unborn'): assert n in _r.__all__, n
test_eq(plural(1, 'file'), '1 file')
test_eq(plural(2, 'file'), '2 files')
assert unborn('0'*40) and not unborn('a'*40)
r = GitRepo.at(mkrepo())
r.decorations()
assert any(k[0] == str(r.root) for k in _CACHE)
invalidate(r.root)
assert not any(k[0] == str(r.root) for k in _CACHE)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()